## Generate content for algorithms 1-5

### Subtask:
For each algorithm (Consistent Hashing, Load Balancing Algorithms, Leaky Bucket & Token Bucket, Bloom Filters, Merkle Trees), create a dedicated markdown section including: Algorithm Name, Concept Explanation, Real-World Analogy, Use Cases & Trade-offs. Also, generate a code cell with a Python simulation.


**Reasoning**:
Start by creating the markdown section and content for the Consistent Hashing algorithm as per the instructions, and then create the code cell for its simulation.



## Consistent Hashing

### Concept Explanation
Consistent hashing is a distributed hashing scheme that minimizes the number of keys that need to be remapped when a hash table is resized. Instead of mapping keys directly to server nodes, it maps both keys and server nodes to positions on a conceptual ring. When a node is added or removed, only the keys that were mapped to that specific node (or a small adjacent section of the ring) need to be remapped to other nodes, rather than a large portion of the keys requiring remapping as in traditional hashing.

### Real-World Analogy
Imagine a circular library where books (data) are placed on shelves (servers) arranged around the circle. When a new shelf is added or removed, you only need to move the books from the immediate vicinity of that shelf to the next available shelf. You don't need to rearrange all the books in the entire library. Consistent hashing works similarly, minimizing data movement when servers are added or removed.

### Use Cases & Trade-offs
**Use Cases:**
*   Distributed caching systems (e.g., Memcached, Redis)
*   Distributed databases (e.g., DynamoDB, Cassandra)
*   Content delivery networks (CDNs)

**Trade-offs:**
*   **Advantages:**
    *   Minimizes data redistribution on node changes.
    *   Better availability and scalability in distributed systems.
*   **Disadvantages:**
    *   Can be more complex to implement than simple hashing.
    *   Requires careful consideration of virtual nodes to ensure even data distribution.


In [2]:
import hashlib

class ConsistentHashing:
    def __init__(self, nodes, replicas=3):
        self.replicas = replicas
        self.nodes = set()
        self.ring = dict()
        self._keys = []
        for node in nodes:
            self.add_node(node)

    def _hash(self, key):
        return int(hashlib.md5(key.encode('utf-8')).hexdigest(), 16)

    def add_node(self, node):
        for i in range(self.replicas):
            key = self._hash(f"{node}:{i}")
            self.ring[key] = node
            self._keys.append(key)
        self._keys.sort()
        self.nodes.add(node)

    def remove_node(self, node):
        for i in range(self.replicas):
            key = self._hash(f"{node}:{i}")
            del self.ring[key]
            self._keys.remove(key)
        self.nodes.remove(node)

    def get_node(self, key):
        if not self.ring:
            return None
        hash_key = self._hash(key)
        # Find the first node on the ring with a hash greater than or equal to the key's hash
        for node_key in self._keys:
            if node_key >= hash_key:
                return self.ring[node_key]
        # If no such node is found, wrap around to the first node
        return self.ring[self._keys[0]]

# Example Usage
nodes = ['nodeA', 'nodeB', 'nodeC']
consistent_hash = ConsistentHashing(nodes)

print("Initial node assignments:")
print(f"Key 'apple' maps to node: {consistent_hash.get_node('apple')}")
print(f"Key 'banana' maps to node: {consistent_hash.get_node('banana')}")
print(f"Key 'cherry' maps to node: {consistent_hash.get_node('cherry')}")

print("\nAdding nodeD...")
consistent_hash.add_node('nodeD')
print(f"Key 'apple' now maps to node: {consistent_hash.get_node('apple')}")
print(f"Key 'banana' now maps to node: {consistent_hash.get_node('banana')}")
print(f"Key 'cherry' now maps to node: {consistent_hash.get_node('cherry')}")
print(f"Key 'date' maps to node: {consistent_hash.get_node('date')}")

print("\nRemoving nodeB...")
consistent_hash.remove_node('nodeB')
print(f"Key 'apple' now maps to node: {consistent_hash.get_node('apple')}")
print(f"Key 'banana' now maps to node: {consistent_hash.get_node('banana')}")
print(f"Key 'cherry' now maps to node: {consistent_hash.get_node('cherry')}")
print(f"Key 'date' now maps to node: {consistent_hash.get_node('date')}")

Initial node assignments:
Key 'apple' maps to node: nodeA
Key 'banana' maps to node: nodeC
Key 'cherry' maps to node: nodeC

Adding nodeD...
Key 'apple' now maps to node: nodeA
Key 'banana' now maps to node: nodeC
Key 'cherry' now maps to node: nodeC
Key 'date' maps to node: nodeC

Removing nodeB...
Key 'apple' now maps to node: nodeA
Key 'banana' now maps to node: nodeC
Key 'cherry' now maps to node: nodeC
Key 'date' now maps to node: nodeC


## Load Balancing Algorithms

### Concept Explanation
Load balancing is the process of distributing network traffic evenly across a group of backend servers, known as a server farm or server pool. This ensures that no single server is overloaded, improving application responsiveness and availability. Different algorithms determine how traffic is distributed, such as Round Robin (distributing requests sequentially to each server in turn) or Least Connections (directing traffic to the server with the fewest active connections).

### Real-World Analogy
Imagine a popular restaurant with multiple cashiers (servers). A host (load balancer) directs customers (requests) to the different cashiers. A Round Robin approach would be sending customers to cashier 1, then 2, then 3, and so on, in rotation. A Least Connections approach would be sending the next customer to the cashier with the shortest line.

### Use Cases & Trade-offs
**Use Cases:**
*   Distributing traffic for web servers and applications
*   Managing connections for databases
*   Balancing workloads in cloud computing environments

**Trade-offs:**
*   **Advantages:**
    *   Improves performance and reduces server response time.
    *   Increases availability and reliability by preventing single points of failure.
    *   Enables scalability by easily adding or removing servers.
*   **Disadvantages:**
    *   Adds complexity to the system architecture.
    *   Can introduce a single point of failure if the load balancer itself fails (addressed by redundant load balancers).
    *   Some algorithms may not distribute load perfectly evenly under all circumstances.


In [4]:
import itertools

class RoundRobinLoadBalancer:
    def __init__(self, servers):
        self.servers = servers
        self._server_cycle = itertools.cycle(self.servers)

    def get_next_server(self):
        return next(self._server_cycle)

# Example Usage
servers = ['server1', 'server2', 'server3']
lb = RoundRobinLoadBalancer(servers)

print("Round Robin Load Balancing:")
for i in range(10):
    print(f"Request {i+1} -> {lb.get_next_server()}")

class LeastConnectionLoadBalancer:
    def __init__(self, servers):
        self.servers = {server: 0 for server in servers}

    def get_next_server(self):
        least_connected_server = min(self.servers, key=self.servers.get)
        self.servers[least_connected_server] += 1
        return least_connected_server

    def release_connection(self, server):
        if server in self.servers and self.servers[server] > 0:
            self.servers[server] -= 1

# Example Usage
servers = ['serverA', 'serverB', 'serverC']
lb_lc = LeastConnectionLoadBalancer(servers)

print("\nLeast Connection Load Balancing:")
for i in range(10):
    server = lb_lc.get_next_server()
    print(f"Request {i+1} -> {server}")
    # Simulate releasing connections after some requests
    if i % 3 == 0 and i > 0:
        lb_lc.release_connection(server)
        print(f"Released connection from {server}")


Round Robin Load Balancing:
Request 1 -> server1
Request 2 -> server2
Request 3 -> server3
Request 4 -> server1
Request 5 -> server2
Request 6 -> server3
Request 7 -> server1
Request 8 -> server2
Request 9 -> server3
Request 10 -> server1

Least Connection Load Balancing:
Request 1 -> serverA
Request 2 -> serverB
Request 3 -> serverC
Request 4 -> serverA
Released connection from serverA
Request 5 -> serverA
Request 6 -> serverB
Request 7 -> serverC
Released connection from serverC
Request 8 -> serverC
Request 9 -> serverA
Request 10 -> serverB
Released connection from serverB


## Leaky Bucket & Token Bucket

### Concept Explanation
Leaky Bucket and Token Bucket are two common algorithms used for rate limiting in computer networks and systems.

*   **Leaky Bucket:** This algorithm smooths out bursts of traffic by allowing data to pass through at a fixed rate, like water leaking from a bucket. If the bucket is full, incoming traffic is discarded. It enforces a strict output rate.
*   **Token Bucket:** This algorithm allows for bursts of traffic up to a certain limit. Tokens are generated at a fixed rate and placed in a "bucket". Each incoming request consumes a token. If the bucket is empty, the request is either dropped or queued until a token is available. It enforces an average output rate but allows for some burstiness.

### Real-World Analogy
*   **Leaky Bucket:** Imagine a bucket with a small hole in the bottom. Water (traffic) can be poured into the bucket quickly, but it can only leak out at a constant rate through the hole. If you pour water in faster than it leaks out, the bucket overflows (traffic is dropped).
*   **Token Bucket:** Imagine a vending machine that requires tokens. Tokens are dispensed into a collection bin (the bucket) at a steady rate. You can insert tokens and get a snack (process a request) as long as there are tokens available. If you run out of tokens, you have to wait until more are dispensed.

### Use Cases & Trade-offs
**Use Cases:**
*   Rate limiting API requests
*   Controlling traffic flow in networks
*   Preventing denial-of-service attacks

**Trade-offs:**
*   **Leaky Bucket:**
    *   **Advantages:** Simple to implement, provides smooth output rate.
    *   **Disadvantages:** Can drop legitimate bursts of traffic, doesn't utilize available bandwidth efficiently during low traffic periods.
*   **Token Bucket:**
    *   **Advantages:** Allows for bursts of traffic, utilizes bandwidth more efficiently than Leaky Bucket.
    *   **Disadvantages:** Can be slightly more complex to implement than Leaky Bucket.


In [6]:
import time

class LeakyBucket:
    def __init__(self, capacity, leak_rate):
        self.capacity = capacity
        self.leak_rate = leak_rate  # units per second
        self.current_level = 0
        self.last_leak_time = time.time()

    def allow_request(self, tokens):
        now = time.time()
        elapsed_time = now - self.last_leak_time
        leaked_amount = elapsed_time * self.leak_rate
        self.current_level = max(0, self.current_level - leaked_amount)
        self.last_leak_time = now

        if self.current_level + tokens <= self.capacity:
            self.current_level += tokens
            return True
        else:
            return False

# Example Usage (Leaky Bucket)
bucket_lb = LeakyBucket(capacity=10, leak_rate=2) # 10 units capacity, 2 units leak per second

print("Leaky Bucket Simulation:")
for i in range(15):
    tokens_needed = 1
    if bucket_lb.allow_request(tokens_needed):
        print(f"Time {time.time():.2f}: Request {i+1} allowed (current level: {bucket_lb.current_level:.2f})")
    else:
        print(f"Time {time.time():.2f}: Request {i+1} denied (bucket full, current level: {bucket_lb.current_level:.2f})")
    time.sleep(0.2) # Simulate time passing between requests


class TokenBucket:
    def __init__(self, capacity, fill_rate):
        self.capacity = capacity
        self.fill_rate = fill_rate # tokens per second
        self.current_tokens = capacity
        self.last_fill_time = time.time()

    def allow_request(self, tokens):
        now = time.time()
        elapsed_time = now - self.last_fill_time
        tokens_to_add = elapsed_time * self.fill_rate
        self.current_tokens = min(self.capacity, self.current_tokens + tokens_to_add)
        self.last_fill_time = now

        if self.current_tokens >= tokens:
            self.current_tokens -= tokens
            return True
        else:
            return False

# Example Usage (Token Bucket)
bucket_tb = TokenBucket(capacity=10, fill_rate=2) # 10 tokens capacity, 2 tokens filled per second

print("\nToken Bucket Simulation:")
for i in range(15):
    tokens_needed = 1
    if bucket_tb.allow_request(tokens_needed):
        print(f"Time {time.time():.2f}: Request {i+1} allowed (current tokens: {bucket_tb.current_tokens:.2f})")
    else:
        print(f"Time {time.time():.2f}: Request {i+1} denied (not enough tokens, current tokens: {bucket_tb.current_tokens:.2f})")
    time.sleep(0.2) # Simulate time passing between requests

Leaky Bucket Simulation:
Time 1760102075.30: Request 1 allowed (current level: 1.00)
Time 1760102075.50: Request 2 allowed (current level: 1.60)
Time 1760102075.70: Request 3 allowed (current level: 2.20)
Time 1760102075.90: Request 4 allowed (current level: 2.80)
Time 1760102076.10: Request 5 allowed (current level: 3.40)
Time 1760102076.30: Request 6 allowed (current level: 4.00)
Time 1760102076.50: Request 7 allowed (current level: 4.60)
Time 1760102076.70: Request 8 allowed (current level: 5.20)
Time 1760102076.90: Request 9 allowed (current level: 5.80)
Time 1760102077.10: Request 10 allowed (current level: 6.40)
Time 1760102077.30: Request 11 allowed (current level: 7.00)
Time 1760102077.50: Request 12 allowed (current level: 7.60)
Time 1760102077.70: Request 13 allowed (current level: 8.20)
Time 1760102077.90: Request 14 allowed (current level: 8.79)
Time 1760102078.10: Request 15 allowed (current level: 9.39)

Token Bucket Simulation:
Time 1760102078.30: Request 1 allowed (curr

## Bloom Filters

### Concept Explanation
A Bloom filter is a space-efficient probabilistic data structure used to test whether an element is a member of a set. It's not guaranteed to be accurate; it can produce false positives (indicating an element is in the set when it's not), but it will never produce false negatives (indicating an element is not in the set when it is). It achieves this by using multiple hash functions to mark bits in a bit array.

### Real-World Analogy
Imagine you have a large collection of books and you want a quick way to check if a specific book *might* be in your collection without searching through every shelf. You could use a system where you have multiple different colored bookmarks (hash functions). When you add a book, you place a specific combination of colored bookmarks on a shared index card (bit array). To check if a book is there, you look at the index card; if the required combination of colored bookmarks isn't there, the book is definitely not in your collection. If the combination *is* there, the book *might* be there (it could be a false positive due to other books using the same bookmark combination).

### Use Cases & Trade-offs
**Use Cases:**
*   Checking for the existence of elements in large datasets where some false positives are acceptable (e.g., spell checkers, caching systems to avoid expensive disk lookups, preventing duplicate entries in databases).
*   Network routers to quickly check if a packet's destination is valid.
*   Filtering out already seen URLs in web crawlers.

**Trade-offs:**
*   **Advantages:**
    *   Very space-efficient compared to traditional hash tables or sets.
    *   Membership queries are very fast.
*   **Disadvantages:**
    *   Can produce false positives.
    *   Elements cannot be removed from the filter.
    *   The probability of false positives increases as more elements are added.


In [8]:
import mmh3
import bitarray

class BloomFilter:
    def __init__(self, capacity, error_rate):
        self.capacity = capacity
        self.error_rate = error_rate
        self.num_bits = self.get_size(capacity, error_rate)
        self.num_hash_functions = self.get_hash_count(self.num_bits, capacity)
        self.bit_array = bitarray.bitarray(self.num_bits, endian='little')
        self.bit_array.setall(0)

    def get_size(self, n, p):
        """Calculates the optimal size of the bit array."""
        m = -(n * self._ln2_squared) / self._ln_p
        return int(m)

    def get_hash_count(self, m, n):
        """Calculates the optimal number of hash functions."""
        k = (m / n) * self._ln2
        return int(k)

    # Precompute constants for efficiency
    _ln2 = 0.6931471805599453
    _ln2_squared = _ln2 * _ln2
    _ln_p = -_ln2_squared # This is incorrect, should be math.log(error_rate)

    def _hashes(self, item):
        """Generates hash values for an item."""
        # Use different seeds for each hash function
        for i in range(self.num_hash_functions):
             # Correct way to calculate ln(p)
            self._ln_p = -math.log(self.error_rate)
            yield mmh3.hash(item, i) % self.num_bits

    def add(self, item):
        """Adds an item to the bloom filter."""
        for index in self._hashes(item):
            self.bit_array[index] = 1

    def contains(self, item):
        """Checks if an item might be in the bloom filter."""
        for index in self._hashes(item):
            if self.bit_array[index] == 0:
                return False
        return True

# Example Usage
import math # Import math for log

# Corrected constants calculation
BloomFilter._ln_p = -math.log(0.01) # Example error rate of 1%

capacity = 1000
error_rate = 0.01
bloom_filter = BloomFilter(capacity, error_rate)

words_to_add = ['apple', 'banana', 'cherry', 'date', 'elderberry']
for word in words_to_add:
    bloom_filter.add(word)

print("Bloom Filter Simulation:")
print(f"Does 'apple' exist? {bloom_filter.contains('apple')}")
print(f"Does 'grape' exist? {bloom_filter.contains('grape')}") # Should be False
print(f"Does 'cherry' exist? {bloom_filter.contains('cherry')}")
print(f"Does 'zucchini' exist? {bloom_filter.contains('zucchini')}") # Should be False (might be a false positive)

# Demonstrate a potential false positive (this is probabilistic and may not happen every time)
# We need to add more elements to increase the chance of a false positive
more_words = [f'word_{i}' for i in range(capacity)]
for word in more_words:
    bloom_filter.add(word)

print("\nAfter adding more words:")
# Now, let's check a word that was not added but might result in a false positive
false_positive_candidate = "not_in_list"
print(f"Does '{false_positive_candidate}' exist? {bloom_filter.contains(false_positive_candidate)}")


ModuleNotFoundError: No module named 'mmh3'

In [9]:
%pip install mmh3 bitarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.3/339.3 kB 18.1 MB/s eta 0:00:00


In [10]:
import mmh3
import bitarray
import math # Import math for log

class BloomFilter:
    def __init__(self, capacity, error_rate):
        self.capacity = capacity
        self.error_rate = error_rate
        self.num_bits = self.get_size(capacity, error_rate)
        self.num_hash_functions = self.get_hash_count(self.num_bits, capacity)
        self.bit_array = bitarray.bitarray(self.num_bits, endian='little')
        self.bit_array.setall(0)

    def get_size(self, n, p):
        """Calculates the optimal size of the bit array."""
        # Correct calculation for optimal size
        m = -(n * math.log(p)) / (math.log(2) ** 2)
        return int(m)

    def get_hash_count(self, m, n):
        """Calculates the optimal number of hash functions."""
        # Correct calculation for optimal number of hash functions
        k = (m / n) * math.log(2)
        return int(k)

    def _hashes(self, item):
        """Generates hash values for an item."""
        # Use different seeds for each hash function
        for i in range(self.num_hash_functions):
            yield mmh3.hash(item, i) % self.num_bits

    def add(self, item):
        """Adds an item to the bloom filter."""
        for index in self._hashes(item):
            self.bit_array[index] = 1

    def contains(self, item):
        """Checks if an item might be in the bloom filter."""
        for index in self._hashes(item):
            if self.bit_array[index] == 0:
                return False
        return True

# Example Usage
capacity = 1000
error_rate = 0.01
bloom_filter = BloomFilter(capacity, error_rate)

words_to_add = ['apple', 'banana', 'cherry', 'date', 'elderberry']
for word in words_to_add:
    bloom_filter.add(word)

print("Bloom Filter Simulation:")
print(f"Does 'apple' exist? {bloom_filter.contains('apple')}")
print(f"Does 'grape' exist? {bloom_filter.contains('grape')}") # Should be False
print(f"Does 'cherry' exist? {bloom_filter.contains('cherry')}")
print(f"Does 'zucchini' exist? {bloom_filter.contains('zucchini')}") # Should be False (might be a false positive)

# Demonstrate a potential false positive (this is probabilistic and may not happen every time)
# We need to add more elements to increase the chance of a false positive
more_words = [f'word_{i}' for i in range(capacity)]
for word in more_words:
    bloom_filter.add(word)

print("\nAfter adding more words:")
# Now, let's check a word that was not added but might result in a false positive
false_positive_candidate = "not_in_list"
print(f"Does '{false_positive_candidate}' exist? {bloom_filter.contains(false_positive_candidate)}")

Bloom Filter Simulation:
Does 'apple' exist? True
Does 'grape' exist? False
Does 'cherry' exist? True
Does 'zucchini' exist? False

After adding more words:
Does 'not_in_list' exist? False


## Merkle Trees

### Concept Explanation
A Merkle tree, also known as a hash tree, is a tree in which every leaf node is labelled with the hash of a data block, and every non-leaf node is labelled with the cryptographic hash of the labels of its child nodes. Merkle trees allow for efficient and secure verification of the contents of large data structures. By checking the Merkle root (the hash at the top of the tree), one can verify if any data block has been tampered with without needing to download the entire dataset.

### Real-World Analogy
Imagine you have a large collection of documents (data blocks) and you want to ensure none of them have been altered. You could take each document and create a unique fingerprint (hash) for it. Then, you pair the fingerprints, create a new fingerprint for each pair, and continue this process up the tree until you have a single master fingerprint (the Merkle root) for the entire collection. If someone changes even a single character in one document, its fingerprint will change, which will change the fingerprint of its parent, and so on, all the way up to the master fingerprint. By comparing the current master fingerprint to a trusted original one, you can quickly tell if anything has been changed in the entire collection without having to re-examine every single document.

### Use Cases & Trade-offs
**Use Cases:**
*   **Blockchain technology:** Used extensively in cryptocurrencies like Bitcoin and Ethereum to efficiently and securely verify transactions.
*   **Distributed file systems:** (e.g., IPFS) to ensure the integrity of data chunks.
*   **Data synchronization and verification:** In systems like Git, to efficiently detect changes and verify file integrity.
*   **Certificate Transparency logs:** To verify the integrity of SSL/TLS certificates.

**Trade-offs:**
*   **Advantages:**
    *   Efficiently verify data integrity without downloading the entire dataset.
    *   Reduces the amount of data needed for verification (only need the Merkle root and the "Merkle path" to the specific data block).
    *   Simplifies the process of proving that a specific data block is part of a larger dataset.
*   **Disadvantages:**
    *   Requires computing and storing hashes for all data blocks and intermediate nodes.
    *   If the Merkle root is compromised, the integrity verification is no longer reliable.
    *   Does not protect against denial-of-service attacks that prevent access to the Merkle root or paths.

**Reasoning**:
Create a code cell with a Python simulation for Merkle Trees, demonstrating the core concept of verifying data integrity using the Merkle root.



In [12]:
import hashlib

def sha256(data):
    """Computes the SHA-256 hash of the input data."""
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

class MerkleTree:
    def __init__(self, data_blocks):
        self.data_blocks = data_blocks
        self.tree = self._build_tree([sha256(block) for block in data_blocks])

    def _build_tree(self, hashes):
        """Recursively builds the Merkle tree from a list of hashes."""
        if len(hashes) == 1:
            return hashes
        new_level = []
        # Pair hashes and compute the hash of their concatenation
        for i in range(0, len(hashes), 2):
            left = hashes[i]
            right = hashes[i+1] if i+1 < len(hashes) else left # Handle odd number of leaves
            new_level.append(sha256(left + right))
        return self._build_tree(new_level)

    def get_root(self):
        """Returns the Merkle root of the tree."""
        return self.tree[0] if self.tree else None

    def get_merkle_path(self, data_block):
        """Generates the Merkle path for a given data block."""
        if data_block not in self.data_blocks:
            return None, "Data block not found."

        index = self.data_blocks.index(data_block)
        current_level = [sha256(block) for block in self.data_blocks]
        path = []

        while len(current_level) > 1:
            new_level = []
            for i in range(0, len(current_level), 2):
                left_hash = current_level[i]
                # Handle odd number of nodes at a level
                right_hash = current_level[i+1] if i+1 < len(current_level) else left_hash

                # Determine if the target hash is the left or right child
                if i == index or i + 1 == index:
                    if i == index:
                        path.append((right_hash, 'right'))
                    else: # i + 1 == index
                        path.append((left_hash, 'left'))

                new_level.append(sha256(left_hash + right_hash))
            current_level = new_level
            index //= 2 # Move up to the parent index

        return path, None

    def verify_block(self, data_block, root_hash, merkle_path):
        """Verifies if a data block is part of the tree with the given root using the Merkle path."""
        current_hash = sha256(data_block)
        for sibling_hash, position in merkle_path:
            if position == 'left':
                current_hash = sha256(sibling_hash + current_hash)
            else: # position == 'right'
                current_hash = sha256(current_hash + sibling_hash)
        return current_hash == root_hash

# Example Usage
data = ["data block 1", "data block 2", "data block 3", "data block 4"]
merkle_tree = MerkleTree(data)
merkle_root = merkle_tree.get_root()

print(f"Merkle Root: {merkle_root}")

# Verify an existing data block
block_to_verify = "data block 2"
merkle_path, error = merkle_tree.get_merkle_path(block_to_verify)

if not error:
    is_valid = merkle_tree.verify_block(block_to_verify, merkle_root, merkle_path)
    print(f"\nVerifying '{block_to_verify}': {is_valid}")

    # Simulate tampering with the data block
    tampered_block = "data block 2 - TAMPERED"
    is_valid_tampered = merkle_tree.verify_block(tampered_block, merkle_root, merkle_path)
    print(f"Verifying tampered '{tampered_block}': {is_valid_tampered}")

    # Verify a non-existent data block
    non_existent_block = "data block 5"
    merkle_path_non_existent, error_non_existent = merkle_tree.get_merkle_path(non_existent_block)
    print(f"\nAttempting to get path for non-existent block: {error_non_existent}")

else:
    print(f"Error getting Merkle path: {error}")


Merkle Root: ee6e3eea43ebe5d50d1021246453e7a81429ddad577b58d5c10e2ebea7c51599

Verifying 'data block 2': True
Verifying tampered 'data block 2 - TAMPERED': False

Attempting to get path for non-existent block: Data block not found.


## Quorum Algorithms

### Concept Explanation
Quorum algorithms are used in distributed systems to ensure consistency and availability when reading or writing data to replicated servers. A quorum is the minimum number of replicas that must acknowledge a read or write operation for it to be considered successful. By requiring a quorum, the system can guarantee that an operation has been seen by a sufficient number of replicas to maintain data consistency, even in the presence of failures. The most common quorum system is the majority quorum, where the quorum size is more than half of the total number of replicas (N/2 + 1). This ensures that any read quorum and any write quorum will overlap, guaranteeing that a read operation will see the result of the most recent successful write operation.

### Real-World Analogy
Imagine a secret ballot election where votes are cast at multiple polling stations (replicas). To ensure the election result is valid and reflects the majority, you don't just count the votes from one station. Instead, you require a majority of polling stations to report their results before declaring a winner. This majority (the quorum) guarantees that even if some stations are late or have issues, the final count is based on a significant portion of the votes, making it highly likely to be accurate and consistent across different observers.

### Use Cases & Trade-offs
**Use Cases:**
*   Distributed databases and key-value stores (e.g., Apache Cassandra, Amazon DynamoDB) for ensuring data consistency across replicas.
*   Distributed coordination services (e.g., Apache ZooKeeper) for managing metadata and configuration.
*   Implementing distributed locks and leader election.

**Trade-offs:**
*   **Advantages:**
    *   Provides tunable consistency and availability guarantees (e.g., strong consistency with majority quorums).
    *   Can tolerate a certain number of replica failures while remaining available.
*   **Disadvantages:**
    *   Higher latency for operations as they need to wait for acknowledgments from a quorum of replicas.
    *   Can reduce availability if the required quorum size is too large and many replicas are unavailable.
    *   Requires careful configuration of read and write quorum sizes to balance consistency and availability needs.


In [14]:
import random
import time

class Replica:
    def __init__(self, id):
        self.id = id
        self.data = None
        self.timestamp = 0
        self.is_available = True

    def write(self, data, timestamp):
        if self.is_available:
            if timestamp > self.timestamp:
                self.data = data
                self.timestamp = timestamp
            return True
        return False

    def read(self):
        if self.is_available:
            return self.data, self.timestamp
        return None, 0

    def set_availability(self, available):
        self.is_available = available

class QuorumSystem:
    def __init__(self, num_replicas, read_quorum_size, write_quorum_size):
        self.num_replicas = num_replicas
        self.read_quorum_size = read_quorum_size
        self.write_quorum_size = write_quorum_size
        self.replicas = [Replica(i) for i in range(num_replicas)]

        if read_quorum_size + write_quorum_size <= num_replicas:
             print("Warning: Read and write quorums do not overlap, consistency might be compromised.")
        if write_quorum_size <= num_replicas // 2:
             print("Warning: Write quorum is not a strict majority, potential for conflicting writes.")


    def perform_write(self, data, timestamp):
        successful_writes = 0
        for replica in self.replicas:
            if replica.write(data, timestamp):
                successful_writes += 1
        return successful_writes >= self.write_quorum_size

    def perform_read(self):
        read_data = []
        for replica in self.replicas:
            data, timestamp = replica.read()
            if data is not None:
                read_data.append((data, timestamp))

        if len(read_data) < self.read_quorum_size:
            return None, "Read quorum not met"

        # Find the most recent data based on timestamp among the read quorum
        latest_data, latest_timestamp = None, 0
        # Sort by timestamp in descending order
        read_data.sort(key=lambda x: x[1], reverse=True)

        # Consider data from the first 'read_quorum_size' available replicas
        quorum_read_data = read_data[:self.read_quorum_size]


        if quorum_read_data:
            latest_data, latest_timestamp = quorum_read_data[0]
            # In a real system, you'd need a conflict resolution strategy
            # if multiple replicas in the quorum return different data for the same timestamp.
            # For this simulation, we'll just take the data from the replica with the highest timestamp.

        return latest_data, None

# Example Usage
num_replicas = 5
# Majority quorum for strong consistency
read_quorum = 3 # N/2 + 1
write_quorum = 3 # N/2 + 1

quorum_system = QuorumSystem(num_replicas, read_quorum, write_quorum)

print(f"Quorum System with {num_replicas} replicas, Read Quorum: {read_quorum}, Write Quorum: {write_quorum}")

# Simulate a write operation
write_success = quorum_system.perform_write("initial data", int(time.time()))
print(f"\nWrite 'initial data' successful: {write_success}")

# Simulate a read operation
read_data, read_error = quorum_system.perform_read()
print(f"Read data: {read_data}, Error: {read_error}")

# Simulate some replicas becoming unavailable
print("\nSetting replica 1 and 3 as unavailable...")
quorum_system.replicas[1].set_availability(False)
quorum_system.replicas[3].set_availability(False)


# Simulate another write operation
write_success_2 = quorum_system.perform_write("updated data", int(time.time()))
print(f"\nWrite 'updated data' successful: {write_success_2}") # Should be False if write quorum is 3

# Simulate another read operation
read_data_2, read_error_2 = quorum_system.perform_read()
print(f"Read data: {read_data_2}, Error: {read_error_2}") # Should be "Read quorum not met" if read quorum is 3


# Make replicas available again and perform write/read
print("\nSetting replicas 1 and 3 back to available...")
quorum_system.replicas[1].set_availability(True)
quorum_system.replicas[3].set_availability(True)


write_success_3 = quorum_system.perform_write("final data", int(time.time()))
print(f"Write 'final data' successful: {write_success_3}")

read_data_3, read_error_3 = quorum_system.perform_read()
print(f"Read data: {read_data_3}, Error: {read_error_3}")


Quorum System with 5 replicas, Read Quorum: 3, Write Quorum: 3

Write 'initial data' successful: True
Read data: initial data, Error: None

Setting replica 1 and 3 as unavailable...

Write 'updated data' successful: True
Read data: initial data, Error: None

Setting replicas 1 and 3 back to available...
Write 'final data' successful: True
Read data: initial data, Error: None


## Leader Election Algorithms

### Concept Explanation
Leader election is a process in distributed systems where a single process (the leader) is chosen from a group of competing processes to coordinate some activity or manage shared resources. The leader is responsible for making decisions, managing consistency, or ensuring that tasks are performed correctly. Various algorithms exist to reliably elect a leader, even when nodes fail or new nodes join the system. Key properties of leader election algorithms include safety (only one leader is elected at a time) and liveness (a leader is eventually elected if possible).

### Real-World Analogy
Imagine a group of hikers lost in the wilderness. To increase their chances of survival, they need one person to take charge, make decisions about direction, allocate tasks, and keep the group organized. A leader election process would be how they decide who that person is. This might involve a vote (like Paxos or Raft), a process where the person with the most experience or loudest voice takes charge (like some simple decentralized algorithms), or a more formal procedure. Once a leader is chosen, the rest of the group follows their direction.

### Use Cases & Trade-offs
**Use Cases:**
*   **Distributed coordination:** Electing a leader to manage shared resources or coordinate tasks (e.g., in distributed databases or message queues).
*   **High availability:** A new leader can be elected if the current leader fails, ensuring the system remains operational.
*   **Consensus protocols:** Leader election is often a component of consensus algorithms like Raft and Paxos.

**Trade-offs:**
*   **Advantages:**
    *   Simplifies coordination in distributed systems by designating a single point of control for certain operations.
    *   Improves fault tolerance by allowing the system to continue operating after a leader failure.
*   **Disadvantages:**
    *   The leader can become a single point of failure if not handled properly.
    *   Leader election can add complexity to the system.
    *   The process of electing a new leader can introduce downtime or latency.
    *   Split-brain scenarios (where multiple nodes believe they are the leader) can occur if the algorithm is not robust.


In [16]:
import time
import random

class Node:
    def __init__(self, id, num_nodes, nodes_list):
        self.id = id
        self.num_nodes = num_nodes
        self.nodes = nodes_list # Reference to the list of all nodes
        self.is_up = True
        self.leader = None

    def send_election_message(self):
        print(f"Node {self.id} starts election.")
        higher_nodes = [node for node in self.nodes if node.id > self.id and node.is_up]
        if not higher_nodes:
            self.declare_leader()
        else:
            for node in higher_nodes:
                # In a real system, this would be network communication
                print(f"Node {self.id} sending election to Node {node.id}")
                node.receive_election_message(self.id)

    def receive_election_message(self, sender_id):
        if self.is_up:
            print(f"Node {self.id} received election from Node {sender_id}")
            # Send an 'OK' message back to the sender
            # In a real system, this would be network communication
            print(f"Node {self.id} sending OK to Node {sender_id}")
            # If this node has a higher ID, it starts its own election
            if self.id > sender_id:
                 self.send_election_message()
        else:
            print(f"Node {self.id} is down, cannot receive election from {sender_id}")


    def declare_leader(self):
        if self.is_up:
            print(f"Node {self.id} declares itself the leader.")
            self.leader = self.id
            # Inform all other nodes about the new leader
            for node in self.nodes:
                if node.is_up:
                     # In a real system, this would be network communication
                    node.receive_leader_message(self.id)

    def receive_leader_message(self, leader_id):
        if self.is_up:
            print(f"Node {self.id} acknowledges Node {leader_id} as the leader.")
            self.leader = leader_id

    def fail(self):
        print(f"Node {self.id} is failing...")
        self.is_up = False
        self.leader = None # Leader might be this node

# Example Usage (Conceptual Bully Algorithm)
num_nodes = 5
nodes = [Node(i, num_nodes, []) for i in range(num_nodes)]
# Give each node a reference to the list of all nodes (for simulation purposes)
for node in nodes:
    node.nodes = nodes

print("Initial state:")
for node in nodes:
    print(f"Node {node.id} is up: {node.is_up}, Leader: {node.leader}")

# Simulate one node starting an election (e.g., node 0)
print("\nSimulating election started by Node 0:")
nodes[0].send_election_message()

# Simulate a node failing
print("\nSimulating Node 4 failing...")
nodes[4].fail()

# Simulate another node starting an election after the failure (e.g., node 1)
print("\nSimulating election started by Node 1 after failure:")
nodes[1].send_election_message()

# You would typically add more logic for timeouts, retries, etc. in a real implementation

Initial state:
Node 0 is up: True, Leader: None
Node 1 is up: True, Leader: None
Node 2 is up: True, Leader: None
Node 3 is up: True, Leader: None
Node 4 is up: True, Leader: None

Simulating election started by Node 0:
Node 0 starts election.
Node 0 sending election to Node 1
Node 1 received election from Node 0
Node 1 sending OK to Node 0
Node 1 starts election.
Node 1 sending election to Node 2
Node 2 received election from Node 1
Node 2 sending OK to Node 1
Node 2 starts election.
Node 2 sending election to Node 3
Node 3 received election from Node 2
Node 3 sending OK to Node 2
Node 3 starts election.
Node 3 sending election to Node 4
Node 4 received election from Node 3
Node 4 sending OK to Node 3
Node 4 starts election.
Node 4 declares itself the leader.
Node 0 acknowledges Node 4 as the leader.
Node 1 acknowledges Node 4 as the leader.
Node 2 acknowledges Node 4 as the leader.
Node 3 acknowledges Node 4 as the leader.
Node 4 acknowledges Node 4 as the leader.
Node 2 sending elec

## Distributed Lock Algorithms

### Concept Explanation
Distributed lock algorithms are used in distributed systems to ensure that only one process or node can access a shared resource or execute a critical section of code at any given time. This is essential for maintaining data consistency and preventing race conditions when multiple independent processes might try to modify the same data concurrently. Unlike locks in a single-process environment, distributed locks must handle network delays, node failures, and partial failures, making them significantly more complex. Algorithms often involve coordinating with a set of nodes or a dedicated lock service to acquire and release the lock.

### Real-World Analogy
Imagine multiple people in different locations trying to edit the same shared document online simultaneously. Without a locking mechanism, their changes could conflict and overwrite each other, resulting in a corrupted document. A distributed lock is like a system where only one person can have "edit access" at a time. Before editing, a person must "acquire the lock." Once they are done, they "release the lock," allowing someone else to acquire it. This ensures that only one person modifies the document at any moment, maintaining consistency.

### Use Cases & Trade-offs
**Use Cases:**
*   Controlling access to shared resources (e.g., databases, files, hardware devices) in a distributed environment.
*   Implementing critical sections in distributed applications to prevent concurrent execution.
*   Ensuring atomicity of operations spanning multiple nodes.
*   Managing unique identifiers or sequences in a distributed system.

**Trade-offs:**
*   **Advantages:**
    *   Ensures mutual exclusion for shared resources in a distributed setting.
    *   Helps maintain data consistency and prevent corruption.
*   **Disadvantages:**
    *   Can be complex to implement correctly, especially considering failure scenarios (network partitions, node crashes).
    *   Poorly implemented locks can lead to deadlocks (where processes are stuck waiting for each other) or livelocks (where processes repeatedly fail to acquire the lock).
    *   Acquiring and releasing locks adds overhead and can impact performance and latency.
    *   Requires a reliable coordination service or mechanism.


In [18]:
import time
import threading

class DistributedLockService:
    def __init__(self):
        self._lock = threading.Lock()
        self._locked_resource = None

    def acquire_lock(self, resource_name, node_id):
        with self._lock:
            if self._locked_resource is None:
                self._locked_resource = (resource_name, node_id)
                print(f"Node {node_id} acquired lock for {resource_name}")
                return True
            else:
                print(f"Node {node_id} failed to acquire lock for {resource_name}. Locked by {self._locked_resource[1]}")
                return False

    def release_lock(self, resource_name, node_id):
        with self._lock:
            if self._locked_resource == (resource_name, node_id):
                self._locked_resource = None
                print(f"Node {node_id} released lock for {resource_name}")
                return True
            else:
                print(f"Node {node_id} cannot release lock for {resource_name}. Not held by this node or already released.")
                return False

# Simulate nodes trying to access a shared resource
def node_task(node_id, lock_service, resource_name, delay):
    print(f"Node {node_id} attempting to access {resource_name}")
    if lock_service.acquire_lock(resource_name, node_id):
        print(f"Node {node_id} working on {resource_name}...")
        time.sleep(delay) # Simulate work
        lock_service.release_lock(resource_name, node_id)
    else:
        print(f"Node {node_id} skipping work on {resource_name} due to lock")

# Example Usage
lock_service = DistributedLockService()
resource_name = "shared_database"

# Create multiple threads representing different nodes
node_threads = []
for i in range(5):
    thread = threading.Thread(target=node_task, args=(i, lock_service, resource_name, random.uniform(0.5, 2.0)))
    node_threads.append(thread)

print("Simulating nodes attempting to acquire a distributed lock:")
for thread in node_threads:
    thread.start()

# Wait for all threads to complete
for thread in node_threads:
    thread.join()

print("\nSimulation finished.")


Simulating nodes attempting to acquire a distributed lock:
Node 0 attempting to access shared_database
Node 0 acquired lock for shared_database
Node 0 working on shared_database...
Node 1 attempting to access shared_database
Node 1 failed to acquire lock for shared_database. Locked by 0
Node 1 skipping work on shared_database due to lock
Node 2 attempting to access shared_database
Node 2 failed to acquire lock for shared_database. Locked by 0
Node 2 skipping work on shared_database due to lock
Node 3 attempting to access shared_database
Node 3 failed to acquire lock for shared_database. Locked by 0
Node 3 skipping work on shared_database due to lock
Node 4 attempting to access shared_database
Node 4 failed to acquire lock for shared_database. Locked by 0
Node 4 skipping work on shared_database due to lock
Node 0 released lock for shared_database

Simulation finished.


## Raft / Paxos

### Concept Explanation
Raft and Paxos are consensus algorithms used in distributed systems to achieve agreement among a set of nodes on a single value or sequence of values, even in the presence of failures (crashes, network partitions). They are fundamental for building fault-tolerant state machines.

*   **Paxos:** A family of algorithms known for its correctness and ability to reach consensus. It's often considered complex to understand and implement. The basic idea involves proposers proposing values, acceptors voting on proposals, and learners deciding the chosen value. It guarantees safety (never reaching an incorrect decision) but its liveness (eventually reaching a decision) can be challenging under certain failure scenarios.
*   **Raft:** Designed to be more understandable than Paxos while achieving the same level of fault tolerance. Raft achieves consensus by electing a prominent leader, who manages a replicated log. Clients interact with the leader, and the leader coordinates log replication to followers. Consensus is reached when an entry is replicated to a majority of followers. Raft simplifies consensus into distinct roles (Leader, Follower, Candidate) and phases (Leader Election, Log Replication).

### Real-World Analogy
Imagine a group of people trying to agree on the date for a meeting (the value to agree on).

*   **Paxos:** Like a very formal and complex negotiation process. Individuals propose dates, others vote on them, and eventually, if enough people agree on a proposal, that date is chosen. The rules are strict to ensure everyone eventually agrees on the same date, even if some people are absent or change their minds.
*   **Raft:** Like a meeting with a designated chairperson (the leader). The chairperson proposes a date and writes it down on a shared agenda (the replicated log). They then ask others to confirm they've written it down. Once a majority confirms, the date is officially decided. If the chairperson leaves, the group holds a quick process to elect a new one. This approach is more structured and easier to follow than the complex negotiation.

### Use Cases & Trade-offs
**Use Cases:**
*   Building fault-tolerant distributed databases and key-value stores.
*   Implementing distributed file systems with strong consistency.
*   Managing cluster metadata and configuration in systems like Kubernetes and etcd.
*   Ensuring consistency in distributed queues and messaging systems.

**Trade-offs:**
*   **Paxos:**
    *   **Advantages:** Highly robust and proven correctness.
    *   **Disadvantages:** Very difficult to understand and implement correctly, can have liveness issues in certain scenarios.
*   **Raft:**
    *   **Advantages:** Designed for understandability, easier to implement and reason about than Paxos, strong leader model simplifies log management.
    *   **Disadvantages:** Leader failure requires re-election, which can introduce temporary unavailability; relies on a strong leader, which can be a bottleneck in some scenarios.

Both algorithms provide strong consistency guarantees but can have performance implications due to the need for coordination and replication across multiple nodes. Choosing between them often comes down to a trade-off between complexity and ease of implementation versus theoretical robustness.


In [20]:
import time
import random

class Node:
    def __init__(self, id, num_nodes):
        self.id = id
        self.num_nodes = num_nodes
        self.current_term = 0
        self.voted_for = None
        self.log = [] # Simplified log
        self.state = "Follower" # States: Follower, Candidate, Leader
        self.leader_id = None
        self.votes_received = set()

    def request_vote(self, candidate_id, term):
        # Simplified vote request logic
        if term > self.current_term:
            self.current_term = term
            self.state = "Follower"
            self.voted_for = candidate_id
            print(f"Node {self.id} votes for Candidate {candidate_id} in term {term}")
            return True, self.current_term
        elif term == self.current_term and self.voted_for is None:
             self.voted_for = candidate_id
             print(f"Node {self.id} votes for Candidate {candidate_id} in term {term}")
             return True, self.current_term
        else:
            print(f"Node {self.id} denies vote for Candidate {candidate_id} in term {term}")
            return False, self.current_term

    def send_heartbeat(self, leader_id, term):
        # Simplified heartbeat logic
        if term >= self.current_term:
            self.current_term = term
            self.state = "Follower"
            self.leader_id = leader_id
            # print(f"Node {self.id} received heartbeat from Leader {leader_id} in term {term}")
            return True, self.current_term
        return False, self.current_term


class RaftSimulation:
    def __init__(self, num_nodes):
        self.num_nodes = num_nodes
        self.nodes = [Node(i, num_nodes) for i in range(num_nodes)]
        self.majority = num_nodes // 2 + 1
        self._network = self._simulate_network # Simplified network interaction

    def _simulate_network(self, sender_id, message_type, **kwargs):
        """Simulates sending messages between nodes."""
        responses = []
        for node in self.nodes:
            if node.id != sender_id:
                if message_type == "RequestVote":
                    vote_granted, term = node.request_vote(kwargs['candidate_id'], kwargs['term'])
                    responses.append((node.id, vote_granted, term))
                elif message_type == "AppendEntries": # Heartbeat is a type of AppendEntries
                    success, term = node.send_heartbeat(kwargs['leader_id'], kwargs['term'])
                    responses.append((node.id, success, term))
        return responses


    def run_election(self, candidate_id):
        """Simulates a node starting an election."""
        candidate_node = self.nodes[candidate_id]
        if candidate_node.state == "Follower":
            candidate_node.state = "Candidate"
            candidate_node.current_term += 1
            candidate_node.voted_for = candidate_id
            candidate_node.votes_received = {candidate_id}
            print(f"\nNode {candidate_id} becomes Candidate in term {candidate_node.current_term}")

            responses = self._network(candidate_id, "RequestVote", candidate_id=candidate_id, term=candidate_node.current_term)

            for responder_id, vote_granted, term in responses:
                if vote_granted:
                    candidate_node.votes_received.add(responder_id)
                    print(f"Node {candidate_id} received vote from Node {responder_id}")
                if term > candidate_node.current_term:
                    candidate_node.current_term = term
                    candidate_node.state = "Follower"
                    print(f"Node {candidate_id} reverts to Follower due to higher term from Node {responder_id}")
                    return # Election failed

            if len(candidate_node.votes_received) >= self.majority:
                candidate_node.state = "Leader"
                candidate_node.leader_id = candidate_id
                print(f"\nNode {candidate_id} elected as Leader in term {candidate_node.current_term}")
                # Leader sends initial heartbeat
                self.send_heartbeats(candidate_id, candidate_node.current_term)
            else:
                candidate_node.state = "Follower" # Election failed to get majority
                print(f"\nNode {candidate_id} election failed in term {candidate_node.current_term}")


    def send_heartbeats(self, leader_id, term):
        """Simulates the leader sending heartbeats."""
        print(f"\nLeader {leader_id} sending heartbeats in term {term}")
        responses = self._network(leader_id, "AppendEntries", leader_id=leader_id, term=term)
        # In a real system, the leader would process responses, handle inconsistencies, etc.
        # This simulation just shows the heartbeats being sent and received.
        for responder_id, success, term_response in responses:
            if term_response > term:
                # Leader is stale, revert to follower (simplified)
                self.nodes[leader_id].state = "Follower"
                self.nodes[leader_id].leader_id = None
                print(f"Leader {leader_id} is stale, reverting to Follower due to higher term from Node {responder_id}")
                return


# Example Usage (Conceptual Raft - focusing on Leader Election)
num_nodes = 5
raft_sim = RaftSimulation(num_nodes)

print(f"Starting Raft simulation with {num_nodes} nodes. Majority needed: {raft_sim.majority}")

# Simulate nodes starting elections (one by one, simplified)
# In a real system, this would happen based on timeouts
raft_sim.run_election(0)
raft_sim.run_election(1)
raft_sim.run_election(2) # This one should become leader if others are followers

print("\nFinal state:")
for node in raft_sim.nodes:
    print(f"Node {node.id}: State={node.state}, Term={node.current_term}, Leader={node.leader_id}")


Starting Raft simulation with 5 nodes. Majority needed: 3

Node 0 becomes Candidate in term 1
Node 1 votes for Candidate 0 in term 1
Node 2 votes for Candidate 0 in term 1
Node 3 votes for Candidate 0 in term 1
Node 4 votes for Candidate 0 in term 1
Node 0 received vote from Node 1
Node 0 received vote from Node 2
Node 0 received vote from Node 3
Node 0 received vote from Node 4

Node 0 elected as Leader in term 1

Leader 0 sending heartbeats in term 1

Node 1 becomes Candidate in term 2
Node 0 votes for Candidate 1 in term 2
Node 2 votes for Candidate 1 in term 2
Node 3 votes for Candidate 1 in term 2
Node 4 votes for Candidate 1 in term 2
Node 1 received vote from Node 0
Node 1 received vote from Node 2
Node 1 received vote from Node 3
Node 1 received vote from Node 4

Node 1 elected as Leader in term 2

Leader 1 sending heartbeats in term 2

Node 2 becomes Candidate in term 3
Node 0 votes for Candidate 2 in term 3
Node 1 votes for Candidate 2 in term 3
Node 3 votes for Candidate 2 i

## Gossip Protocol

### Concept Explanation
The Gossip Protocol is a decentralized communication protocol where nodes in a distributed system periodically exchange information with a small, randomly selected set of other nodes. This process is repeated, allowing information to spread throughout the network in a way that resembles the spread of gossip or an epidemic. There are variations like "push" (send information), "pull" (request information), or "push-pull" (both send and request). It's eventually consistent, meaning all nodes will eventually receive the information, but there's no guarantee about when.

### Real-World Analogy
Imagine a rumor spreading through a town. Someone hears a piece of news (information). They then tell a few friends (exchange with random nodes). Those friends then tell a few of their friends, and so on. Eventually, the rumor (information) spreads throughout the entire town (network). It's not instantaneous, and some people might hear it before others, but given enough time, most people will eventually hear the news.

### Use Cases & Trade-offs
**Use Cases:**
*   **Failure detection:** Nodes can gossip about the health status of other nodes to quickly identify failures (e.g., in Cassandra, Consul).
*   **Membership management:** Nodes can learn about new nodes joining or existing nodes leaving the cluster.
*   **Data dissemination:** Spreading configuration updates or other information throughout a large cluster.
*   **Achieving eventual consistency:** Used in databases and other systems where strong consistency is not strictly required for every operation.

**Trade-offs:**
*   **Advantages:**
    *   Highly robust and fault-tolerant; information continues to spread even if some nodes fail.
    *   Scalable; the communication overhead per node is relatively low, regardless of the total number of nodes.
    *   Simple to implement compared to some other distributed protocols.
    *   Decentralized; no single point of failure for the communication mechanism itself.
*   **Disadvantages:**
    *   Eventually consistent; there's a delay before information propagates to all nodes.
    *   Can generate significant network traffic in large or highly dynamic networks.
    *   No strong guarantees about the speed of propagation or the order in which nodes receive information.
    *   Not suitable for applications requiring strict immediate consistency.


In [22]:
import time
import random

class GossipNode:
    def __init__(self, id, all_nodes):
        self.id = id
        self.all_nodes = all_nodes # Reference to all nodes in the simulation
        self.state = {"status": "healthy", "version": 0} # Example state
        self.known_states = {node.id: {"status": "unknown", "version": -1} for node in all_nodes}
        self.known_states[self.id] = self.state # Node knows its own state

    def update_state(self, new_status):
        self.state["status"] = new_status
        self.state["version"] += 1
        self.known_states[self.id] = self.state
        print(f"Node {self.id} updated its state to {self.state['status']} (Version {self.state['version']})")


    def gossip(self, num_peers=2):
        """Simulates a node gossiping with a few random peers."""
        available_peers = [node for node in self.all_nodes if node.id != self.id]
        if not available_peers:
            return

        peers_to_gossip_with = random.sample(available_peers, min(num_peers, len(available_peers)))

        # Simulate Push-Pull gossip
        for peer in peers_to_gossip_with:
            print(f"Node {self.id} gossiping with Node {peer.id}")
            # Push: Send own state and states known to be newer than peer's
            states_to_push = {
                node_id: state for node_id, state in self.known_states.items()
                if state["version"] > peer.known_states.get(node_id, {"version": -1})["version"]
            }
            peer.receive_gossip_push(self.id, states_to_push)

            # Pull: Request states peer knows that are newer than own
            states_to_request = {
                 node_id: self.known_states.get(node_id, {"version": -1})["version"]
                 for node_id in peer.known_states.keys()
                 if peer.known_states[node_id]["version"] > self.known_states.get(node_id, {"version": -1})["version"]
            }
            if states_to_request:
                 peer.receive_gossip_pull_request(self.id, states_to_request)


    def receive_gossip_push(self, sender_id, received_states):
        """Handles incoming gossip (push part)."""
        updated_count = 0
        for node_id, state in received_states.items():
            if state["version"] > self.known_states.get(node_id, {"version": -1})["version"]:
                self.known_states[node_id] = state
                updated_count += 1
                print(f"Node {self.id} received newer state for Node {node_id} (Version {state['version']}) from Node {sender_id}")
        # In a real system, you might immediately gossip about newly learned states


    def receive_gossip_pull_request(self, sender_id, requested_states):
        """Handles incoming gossip pull requests."""
        states_to_send_back = {}
        for node_id, version_known_by_sender in requested_states.items():
            if node_id in self.known_states and self.known_states[node_id]["version"] > version_known_by_sender:
                states_to_send_back[node_id] = self.known_states[node_id]

        if states_to_send_back:
            print(f"Node {self.id} sending requested states to Node {sender_id}")
            # In a real system, this would be a separate response message
            self.all_nodes[sender_id].receive_gossip_push(self.id, states_to_send_back)


# Example Usage
num_nodes = 5
# Create nodes with a reference to the list of all nodes
nodes = [GossipNode(i, []) for i in range(num_nodes)]
for node in nodes:
    node.all_nodes = nodes # Provide the reference

print(f"Starting Gossip Protocol simulation with {num_nodes} nodes.")

# Simulate one node updating its state
nodes[0].update_state("down")

print("\nInitial known states:")
for node in nodes:
    print(f"Node {node.id} knows: {node.known_states}")

# Simulate gossip rounds
num_gossip_rounds = 5
print(f"\nSimulating {num_gossip_rounds} gossip rounds...")

for round_num in range(num_gossip_rounds):
    print(f"\n--- Gossip Round {round_num + 1} ---")
    for node in nodes:
        node.gossip()
    time.sleep(0.5) # Simulate time passing between rounds

print("\nFinal known states:")
for node in nodes:
    print(f"Node {node.id} knows: {node.known_states}")

# Check if the updated state has propagated to all nodes
updated_state_propagated = all(node.known_states.get(0, {}).get("status") == "down" for node in nodes)
print(f"\nHas Node 0's 'down' state propagated to all nodes? {updated_state_propagated}")


Starting Gossip Protocol simulation with 5 nodes.
Node 0 updated its state to down (Version 1)

Initial known states:
Node 0 knows: {0: {'status': 'down', 'version': 1}}
Node 1 knows: {1: {'status': 'healthy', 'version': 0}}
Node 2 knows: {2: {'status': 'healthy', 'version': 0}}
Node 3 knows: {3: {'status': 'healthy', 'version': 0}}
Node 4 knows: {4: {'status': 'healthy', 'version': 0}}

Simulating 5 gossip rounds...

--- Gossip Round 1 ---
Node 0 gossiping with Node 4
Node 4 received newer state for Node 0 (Version 1) from Node 0
Node 4 sending requested states to Node 0
Node 0 received newer state for Node 4 (Version 0) from Node 4
Node 0 gossiping with Node 2
Node 2 received newer state for Node 0 (Version 1) from Node 0
Node 2 received newer state for Node 4 (Version 0) from Node 0
Node 2 sending requested states to Node 0
Node 0 received newer state for Node 2 (Version 0) from Node 2
Node 1 gossiping with Node 0
Node 0 received newer state for Node 1 (Version 0) from Node 1
Node 0

## Vector Clocks / Lamport Timestamps

### Concept Explanation
Vector Clocks and Lamport Timestamps are mechanisms used in distributed systems to establish a partial ordering of events, which is crucial for understanding causality in systems where events happen concurrently across different processes or nodes.

*   **Lamport Timestamps:** Assigns a single scalar timestamp to each event. When a process sends a message, it includes its current timestamp. When a process receives a message, it updates its timestamp to be the maximum of its current timestamp and the received timestamp, plus one. This provides a *happened-before* relationship: if event A happened before event B in the same process, or if A is the sending of a message and B is the receiving of that message, then A happened before B. However, Lamport timestamps cannot tell if two events are concurrent.
*   **Vector Clocks:** Assigns a vector (an array or list) of timestamps to each event, with one entry for each process in the system. When a process experiences an event, it increments its own entry in its vector clock. When a process sends a message, it includes its current vector clock. When a process receives a message, it updates its vector clock by taking the maximum of each entry in its own vector and the received vector, and then increments its own entry. Vector clocks can determine if two events are concurrent or if one happened before the other.

### Real-World Analogy
*   **Lamport Timestamps:** Imagine a group of friends writing letters to each other. Each letter has a simple sequential number (the timestamp). When you receive a letter with a higher number than your last received letter, you know that letter was sent *after* the previous one you received from that friend. You also increment your own number for the next letter you send. This helps you understand the order of letters from a specific friend, but if you receive letters from two different friends, the numbers alone don't tell you which friend's letter was written first overall.
*   **Vector Clocks:** Imagine the same group of friends, but this time each letter has a list of numbers, one for each friend. The list shows how many letters the sender has received from *each* friend when they wrote that letter. When you receive a letter, you update your list based on the sender's list (taking the higher number for each friend) and then increment your own count before sending your next letter. This allows you to see not just the order from one friend, but the relative order of events across all friends – you can tell if one letter was definitely written after another, or if they were written independently (concurrently) without knowledge of each other.

### Use Cases & Trade-offs
**Use Cases:**
*   **Determining causality:** Understanding the causal relationships between events in distributed systems, which is essential for debugging and ensuring correctness.
*   **Conflict resolution:** In eventually consistent systems, vector clocks can help detect conflicting updates that happened concurrently.
*   **Garbage collection:** In distributed systems, vector clocks can help determine when an object is no longer reachable by any process.
*   **Ensuring consistency models:** Implementing consistency models like causal consistency.

**Trade-offs:**
*   **Lamport Timestamps:**
    *   **Advantages:** Simple to implement, requires minimal overhead (a single integer per event).
    *   **Disadvantages:** Cannot detect concurrency; only provides a partial order (happened-before).
*   **Vector Clocks:**
    *   **Advantages:** Can determine causality and concurrency; provides a more precise ordering of events.
    *   **Disadvantages:** More complex to implement than Lamport timestamps; the size of the vector grows with the number of processes, which can be a scalability issue in very large systems.


In [24]:
import threading
import time

class LamportClock:
    def __init__(self, process_id):
        self.process_id = process_id
        self.time = 0

    def event(self):
        """Simulates an internal event."""
        self.time += 1
        print(f"Process {self.process_id}: Internal event, time = {self.time}")

    def send(self, message):
        """Simulates sending a message."""
        self.time += 1
        print(f"Process {self.process_id}: Sending message '{message}' with time = {self.time}")
        return {"message": message, "timestamp": self.time}

    def receive(self, received_message):
        """Simulates receiving a message."""
        received_time = received_message["timestamp"]
        self.time = max(self.time, received_time) + 1
        print(f"Process {self.process_id}: Received message '{received_message['message']}', updated time = {self.time}")


class VectorClock:
    def __init__(self, process_id, num_processes):
        self.process_id = process_id
        self.num_processes = num_processes
        self.clock = [0] * num_processes

    def event(self):
        """Simulates an internal event."""
        self.clock[self.process_id] += 1
        print(f"Process {self.process_id}: Internal event, clock = {self.clock}")

    def send(self, message):
        """Simulates sending a message."""
        self.clock[self.process_id] += 1
        print(f"Process {self.process_id}: Sending message '{message}' with clock = {self.clock}")
        return {"message": message, "clock": list(self.clock)} # Send a copy

    def receive(self, received_message):
        """Simulates receiving a message."""
        received_clock = received_message["clock"]
        for i in range(self.num_processes):
            self.clock[i] = max(self.clock[i], received_clock[i])
        self.clock[self.process_id] += 1
        print(f"Process {self.process_id}: Received message '{received_message['message']}', updated clock = {self.clock}")

# Example Usage: Lamport Timestamps
print("---
 Lamport Timestamps Simulation ---")
process_a_lt = LamportClock(0)
process_b_lt = LamportClock(1)

process_a_lt.event()
message_ab = process_a_lt.send("Hello from A")
process_b_lt.receive(message_ab)

process_b_lt.event()
message_ba = process_b_lt.send("Hello from B")
process_a_lt.receive(message_ba)

process_a_lt.event()

print("\n--- Vector Clocks Simulation ---")
num_processes_vc = 3
process_p0_vc = VectorClock(0, num_processes_vc)
process_p1_vc = VectorClock(1, num_processes_vc)
process_p2_vc = VectorClock(2, num_processes_vc)

# Simulate events and message passing
process_p0_vc.event() # P0: [1, 0, 0]
message_0_to_1 = process_p0_vc.send("Msg 1 from P0") # P0: [2, 0, 0]
process_p1_vc.receive(message_0_to_1) # P1 receives [2, 0, 0], updates to [2, 1, 0]

process_p1_vc.event() # P1: [2, 2, 0]
message_1_to_2 = process_p1_vc.send("Msg 1 from P1") # P1: [2, 3, 0]
process_p2_vc.receive(message_1_to_2) # P2 receives [2, 3, 0], updates to [2, 3, 1]

process_p2_vc.event() # P2: [2, 3, 2]
message_2_to_0 = process_p2_vc.send("Msg 1 from P2") # P2: [2, 3, 3]
process_p0_vc.receive(message_2_to_0) # P0 receives [2, 3, 3], updates to [max(2,2), max(0,3), max(0,3)] + [1,0,0] = [2, 3, 3] + [1,0,0] = [3, 3, 3]

print("\nFinal Vector Clocks:")
print(f"Process 0: {process_p0_vc.clock}")
print(f"Process 1: {process_p1_vc.clock}")
print(f"Process 2: {process_p2_vc.clock}")

# Demonstrate checking causality/concurrency with Vector Clocks
# Event A: P0 sends "Msg 1 from P0" (clock [2, 0, 0])
# Event B: P1 sends "Msg 1 from P1" (clock [2, 3, 0])
# Event C: P2 sends "Msg 1 from P2" (clock [2, 3, 3])

event_a_clock = [2, 0, 0]
event_b_clock = [2, 3, 0]
event_c_clock = [2, 3, 3]

def happens_before(clock1, clock2):
    """Checks if clock1 happens before clock2."""
    # clock1 happens before clock2 if all entries in clock1 are <= corresponding entries in clock2,
    # and at least one entry is strictly less.
    return all(clock1[i] <= clock2[i] for i in range(len(clock1))) and any(clock1[i] < clock2[i] for i in range(len(clock1)))

def are_concurrent(clock1, clock2):
    """Checks if clock1 and clock2 are concurrent."""
    return not happens_before(clock1, clock2) and not happens_before(clock2, clock1)

print("\nCausality/Concurrency Check with Vector Clocks:")
print(f"Does Event A ([2, 0, 0]) happen before Event B ([2, 3, 0])? {happens_before(event_a_clock, event_b_clock)}") # True
print(f"Does Event B ([2, 3, 0]) happen before Event A ([2, 0, 0])? {happens_before(event_b_clock, event_a_clock)}") # False
print(f"Are Event A ([2, 0, 0]) and Event B ([2, 3, 0]) concurrent? {are_concurrent(event_a_clock, event_b_clock)}") # False

print(f"Does Event A ([2, 0, 0]) happen before Event C ([2, 3, 3])? {happens_before(event_a_clock, event_c_clock)}") # True
print(f"Does Event C ([2, 3, 3]) happen before Event A ([2, 0, 0])? {happens_before(event_c_clock, event_a_clock)}") # False
print(f"Are Event A ([2, 0, 0]) and Event C ([2, 3, 3]) concurrent? {are_concurrent(event_a_clock, event_c_clock)}") # False

# Consider two events that haven't directly communicated through a causal chain
# Let's add an internal event on P1 before receiving message from P0
print("\n--- Vector Clocks Simulation (Concurrent Events) ---")
num_processes_vc_2 = 2
process_cx0 = VectorClock(0, num_processes_vc_2)
process_cx1 = VectorClock(1, num_processes_vc_2)

# P0 event
process_cx0.event() # CX0: [1, 0]
msg_cx0 = process_cx0.send("Msg from CX0") # CX0: [2, 0]

# P1 event (concurrent with P0's actions before receiving msg)
process_cx1.event() # CX1: [0, 1]
msg_cx1 = process_cx1.send("Msg from CX1") # CX1: [0, 2]

# P1 receives msg from P0
process_cx1.receive(msg_cx0) # CX1 receives [2, 0], updates to [max(0,2), max(2,0)] + [0,1] = [2, 2] + [0,1] = [2, 3]

# P0 receives msg from P1
process_cx0.receive(msg_cx1) # CX0 receives [0, 2], updates to [max(2,0), max(0,2)] + [1,0] = [2, 2] + [1,0] = [3, 2]


print("\nFinal Vector Clocks (Concurrent Example):")
print(f"Process 0: {process_cx0.clock}")
print(f"Process 1: {process_cx1.clock}")

# Event D: P0 sends "Msg from CX0" (clock [2, 0])
# Event E: P1 sends "Msg from CX1" (clock [0, 2])

event_d_clock = [2, 0]
event_e_clock = [0, 2]

print("\nCausality/Concurrency Check (Concurrent Example):")
print(f"Does Event D ([2, 0]) happen before Event E ([0, 2])? {happens_before(event_d_clock, event_e_clock)}") # False
print(f"Does Event E ([0, 2]) happen before Event D ([2, 0])? {happens_before(event_e_clock, event_d_clock)}") # False
print(f"Are Event D ([2, 0]) and Event E ([0, 2]) concurrent? {are_concurrent(event_d_clock, event_e_clock)}") # True


SyntaxError: unterminated string literal (detected at line 53) (ipython-input-2796653833.py, line 53)

In [25]:
import threading
import time

class LamportClock:
    def __init__(self, process_id):
        self.process_id = process_id
        self.time = 0

    def event(self):
        """Simulates an internal event."""
        self.time += 1
        print(f"Process {self.process_id}: Internal event, time = {self.time}")

    def send(self, message):
        """Simulates sending a message."""
        self.time += 1
        print(f"Process {self.process_id}: Sending message '{message}' with time = {self.time}")
        return {"message": message, "timestamp": self.time}

    def receive(self, received_message):
        """Simulates receiving a message."""
        received_time = received_message["timestamp"]
        self.time = max(self.time, received_time) + 1
        print(f"Process {self.process_id}: Received message '{received_message['message']}', updated time = {self.time}")


class VectorClock:
    def __init__(self, process_id, num_processes):
        self.process_id = process_id
        self.num_processes = num_processes
        self.clock = [0] * num_processes

    def event(self):
        """Simulates an internal event."""
        self.clock[self.process_id] += 1
        print(f"Process {self.process_id}: Internal event, clock = {self.clock}")

    def send(self, message):
        """Simulates sending a message."""
        self.clock[self.process_id] += 1
        print(f"Process {self.process_id}: Sending message '{message}' with clock = {self.clock}")
        return {"message": message, "clock": list(self.clock)} # Send a copy

    def receive(self, received_message):
        """Simulates receiving a message."""
        received_clock = received_message["clock"]
        for i in range(self.num_processes):
            self.clock[i] = max(self.clock[i], received_clock[i])
        self.clock[self.process_id] += 1
        print(f"Process {self.process_id}: Received message '{received_message['message']}', updated clock = {self.clock}")

# Example Usage: Lamport Timestamps
print("--- Lamport Timestamps Simulation ---")
process_a_lt = LamportClock(0)
process_b_lt = LamportClock(1)

process_a_lt.event()
message_ab = process_a_lt.send("Hello from A")
process_b_lt.receive(message_ab)

process_b_lt.event()
message_ba = process_b_lt.send("Hello from B")
process_a_lt.receive(message_ba)

process_a_lt.event()

print("\n--- Vector Clocks Simulation ---")
num_processes_vc = 3
process_p0_vc = VectorClock(0, num_processes_vc)
process_p1_vc = VectorClock(1, num_processes_vc)
process_p2_vc = VectorClock(2, num_processes_vc)

# Simulate events and message passing
process_p0_vc.event() # P0: [1, 0, 0]
message_0_to_1 = process_p0_vc.send("Msg 1 from P0") # P0: [2, 0, 0]
process_p1_vc.receive(message_0_to_1) # P1 receives [2, 0, 0], updates to [2, 1, 0]

process_p1_vc.event() # P1: [2, 2, 0]
message_1_to_2 = process_p1_vc.send("Msg 1 from P1") # P1: [2, 3, 0]
process_p2_vc.receive(message_1_to_2) # P2 receives [2, 3, 0], updates to [2, 3, 1]

process_p2_vc.event() # P2: [2, 3, 2]
message_2_to_0 = process_p2_vc.send("Msg 1 from P2") # P2: [2, 3, 3]
process_p0_vc.receive(message_2_to_0) # P0 receives [2, 3, 3], updates to [max(2,2), max(0,3), max(0,3)] + [1,0,0] = [2, 3, 3] + [1,0,0] = [3, 3, 3]

print("\nFinal Vector Clocks:")
print(f"Process 0: {process_p0_vc.clock}")
print(f"Process 1: {process_p1_vc.clock}")
print(f"Process 2: {process_p2_vc.clock}")

# Demonstrate checking causality/concurrency with Vector Clocks
# Event A: P0 sends "Msg 1 from P0" (clock [2, 0, 0])
# Event B: P1 sends "Msg 1 from P1" (clock [2, 3, 0])
# Event C: P2 sends "Msg 1 from P2" (clock [2, 3, 3])

event_a_clock = [2, 0, 0]
event_b_clock = [2, 3, 0]
event_c_clock = [2, 3, 3]

def happens_before(clock1, clock2):
    """Checks if clock1 happens before clock2."""
    # clock1 happens before clock2 if all entries in clock1 are <= corresponding entries in clock2,
    # and at least one entry is strictly less.
    return all(clock1[i] <= clock2[i] for i in range(len(clock1))) and any(clock1[i] < clock2[i] for i in range(len(clock1)))

def are_concurrent(clock1, clock2):
    """Checks if clock1 and clock2 are concurrent."""
    return not happens_before(clock1, clock2) and not happens_before(clock2, clock1)

print("\nCausality/Concurrency Check with Vector Clocks:")
print(f"Does Event A ([2, 0, 0]) happen before Event B ([2, 3, 0])? {happens_before(event_a_clock, event_b_clock)}") # True
print(f"Does Event B ([2, 3, 0]) happen before Event A ([2, 0, 0])? {happens_before(event_b_clock, event_a_clock)}") # False
print(f"Are Event A ([2, 0, 0]) and Event B ([2, 3, 0]) concurrent? {are_concurrent(event_a_clock, event_b_clock)}") # False

print(f"Does Event A ([2, 0, 0]) happen before Event C ([2, 3, 3])? {happens_before(event_a_clock, event_c_clock)}") # True
print(f"Does Event C ([2, 3, 3]) happen before Event A ([2, 0, 0])? {happens_before(event_c_clock, event_a_clock)}") # False
print(f"Are Event A ([2, 0, 0]) and Event C ([2, 3, 3]) concurrent? {are_concurrent(event_a_clock, event_c_clock)}") # False

# Consider two events that haven't directly communicated through a causal chain
# Let's add an internal event on P1 before receiving message from P0
print("\n--- Vector Clocks Simulation (Concurrent Events) ---")
num_processes_vc_2 = 2
process_cx0 = VectorClock(0, num_processes_vc_2)
process_cx1 = VectorClock(1, num_processes_vc_2)

# P0 event
process_cx0.event() # CX0: [1, 0]
msg_cx0 = process_cx0.send("Msg from CX0") # CX0: [2, 0]

# P1 event (concurrent with P0's actions before receiving msg)
process_cx1.event() # CX1: [0, 1]
msg_cx1 = process_cx1.send("Msg from CX1") # CX1: [0, 2]

# P1 receives msg from P0
process_cx1.receive(msg_cx0) # CX1 receives [2, 0], updates to [max(0,2), max(2,0)] + [0,1] = [2, 2] + [0,1] = [2, 3]

# P0 receives msg from P1
process_cx0.receive(msg_cx1) # CX0 receives [0, 2], updates to [max(2,0), max(0,2)] + [1,0] = [2, 2] + [1,0] = [3, 2]


print("\nFinal Vector Clocks (Concurrent Example):")
print(f"Process 0: {process_cx0.clock}")
print(f"Process 1: {process_cx1.clock}")

# Event D: P0 sends "Msg from CX0" (clock [2, 0])
# Event E: P1 sends "Msg from CX1" (clock [0, 2])

event_d_clock = [2, 0]
event_e_clock = [0, 2]

print("\nCausality/Concurrency Check (Concurrent Example):")
print(f"Does Event D ([2, 0]) happen before Event E ([0, 2])? {happens_before(event_d_clock, event_e_clock)}") # False
print(f"Does Event E ([0, 2]) happen before Event D ([2, 0])? {happens_before(event_e_clock, event_d_clock)}") # False
print(f"Are Event D ([2, 0]) and Event E ([0, 2]) concurrent? {are_concurrent(event_d_clock, event_e_clock)}") # True


--- Lamport Timestamps Simulation ---
Process 0: Internal event, time = 1
Process 0: Sending message 'Hello from A' with time = 2
Process 1: Received message 'Hello from A', updated time = 3
Process 1: Internal event, time = 4
Process 1: Sending message 'Hello from B' with time = 5
Process 0: Received message 'Hello from B', updated time = 6
Process 0: Internal event, time = 7

--- Vector Clocks Simulation ---
Process 0: Internal event, clock = [1, 0, 0]
Process 0: Sending message 'Msg 1 from P0' with clock = [2, 0, 0]
Process 1: Received message 'Msg 1 from P0', updated clock = [2, 1, 0]
Process 1: Internal event, clock = [2, 2, 0]
Process 1: Sending message 'Msg 1 from P1' with clock = [2, 3, 0]
Process 2: Received message 'Msg 1 from P1', updated clock = [2, 3, 1]
Process 2: Internal event, clock = [2, 3, 2]
Process 2: Sending message 'Msg 1 from P2' with clock = [2, 3, 3]
Process 0: Received message 'Msg 1 from P2', updated clock = [3, 3, 3]

Final Vector Clocks:
Process 0: [3, 3, 

## Two-Phase / Three-Phase Commit

### Concept Explanation
Two-Phase Commit (2PC) and Three-Phase Commit (3PC) are distributed consensus algorithms used to ensure that all nodes in a distributed transaction either commit (save) a transaction or abort (discard) it, guaranteeing atomicity across distributed data stores.

*   **Two-Phase Commit (2PC):** Involves a coordinator node and multiple participant nodes. It has two phases:
    1.  **Prepare Phase (Voting Phase):** The coordinator sends a "prepare" request to all participants. Each participant decides if it can commit the transaction and responds with either "yes" (ready to commit) or "no" (cannot commit).
    2.  **Commit Phase (Decision Phase):** If the coordinator receives "yes" from *all* participants, it sends a "commit" command to all participants. If it receives any "no" or a timeout, it sends an "abort" command to all participants. Participants then either commit or abort the transaction locally based on the coordinator's command.
    *   **Problem:** 2PC is blocking. If the coordinator fails *after* sending "prepare" but *before* sending "commit" or "abort", participants who voted "yes" are stuck waiting indefinitely, unable to either commit or abort the transaction.

*   **Three-Phase Commit (3PC):** Designed to be non-blocking under certain network conditions, addressing the blocking problem of 2PC. It adds a third phase:
    1.  **Prepare Phase:** Same as 2PC. Coordinator sends "prepare". Participants respond "yes" or "no".
    2.  **Pre-commit Phase:** If the coordinator received "yes" from all participants, it sends a "pre-commit" command. Participants receive "pre-commit" and acknowledge receipt. At this point, participants know that *all* other participants are ready to commit and that a commit decision is imminent unless a failure occurs.
    3.  **Commit Phase:** After receiving acknowledgments from all participants for the "pre-commit", the coordinator sends the "commit" command. Participants commit the transaction.
    *   **Still problematic:** 3PC is non-blocking *only if* there are no network partitions. If a network partition occurs, it can still lead to inconsistencies (e.g., some nodes commit, others abort). It is also more complex and has higher latency than 2PC.

### Real-World Analogy
Imagine a group of friends deciding whether to go to a movie or stay home.

*   **Two-Phase Commit:** One friend is the decision-maker (coordinator). They ask everyone, "Are you ready to go to the movie?" (Prepare). If *everyone* says "yes", the decision-maker says, "Okay, let's go to the movie!" (Commit). If even one person says "no" or doesn't respond, the decision-maker says, "Okay, we'll stay home." (Abort). If the decision-maker asks, everyone says "yes", but then the decision-maker suddenly becomes unreachable before telling everyone to go, the friends who said "yes" are left waiting, unsure what to do.
*   **Three-Phase Commit:** The decision-maker asks, "Are you ready to go?" (Prepare). If all say "yes", the decision-maker then says, "Okay, we're *almost* going, everyone acknowledge you heard this!" (Pre-commit). Everyone confirms they heard. Finally, the decision-maker says, "Okay, let's *definitely* go!" (Commit). This is better because once you hear "pre-commit", you know everyone else is on board, so even if the decision-maker disappears after that, you have a stronger idea that the group will proceed. However, if the group gets split up (network partition), some might think they are going while others think they are not.

### Use Cases & Trade-offs
**Use Cases:**
*   **Distributed transactions:** Ensuring atomic updates across multiple independent databases or services (though often discouraged in modern systems due to complexity and performance issues).
*   **Coordinating state changes:** Synchronizing state changes across distributed components.

**Trade-offs:**
*   **Two-Phase Commit (2PC):**
    *   **Advantages:** Relatively simpler to understand and implement than 3PC.
    *   **Disadvantages:** Blocking; vulnerable to coordinator failure, leading to potential system unavailability or manual intervention. High latency.
*   **Three-Phase Commit (3PC):**
    *   **Advantages:** Non-blocking under network *delay* (but not partition); theoretically improves availability over 2PC in some failure scenarios.
    *   **Disadvantages:** More complex than 2PC; higher latency; does *not* handle network partitions safely, can lead to inconsistencies; less commonly implemented in practice compared to 2PC or other consensus protocols like Raft/Paxos for strong consistency.

Both protocols can significantly impact performance due to the multiple rounds of communication required. Modern distributed systems often favor other approaches like eventual consistency with conflict resolution (e.g., CRDTs) or consensus protocols (Raft/Paxos) for strong consistency, especially in the presence of network partitions.



In [27]:
import time
import random

class Participant:
    def __init__(self, id):
        self.id = id
        self.state = "initial" # initial, ready, committed, aborted
        self.can_commit = True # Simulate if the participant is willing/able to commit

    def prepare(self):
        if self.can_commit:
            print(f"Participant {self.id}: Received PREPARE. Voting YES.")
            self.state = "ready"
            return True
        else:
            print(f"Participant {self.id}: Received PREPARE. Voting NO (simulated failure/unwillingness).")
            self.state = "aborted"
            return False

    def commit(self):
        if self.state == "ready":
            print(f"Participant {self.id}: Received COMMIT. Committing transaction.")
            self.state = "committed"
            return True
        print(f"Participant {self.id}: Received COMMIT but not in READY state. Current state: {self.state}")
        return False


    def abort(self):
        if self.state != "committed":
            print(f"Participant {self.id}: Received ABORT. Aborting transaction.")
            self.state = "aborted"
            return True
        print(f"Participant {self.id}: Received ABORT but already committed.")
        return False

    def set_can_commit(self, can_commit):
        self.can_commit = can_commit
        if not can_commit:
            print(f"Participant {self.id} is now set to vote NO.")

class Coordinator:
    def __init__(self, participants):
        self.participants = participants
        self.transaction_state = "initial" # initial, preparing, committed, aborted

    def start_transaction(self):
        print("\nCoordinator: Starting transaction.")
        self.transaction_state = "preparing"
        votes = []
        print("Coordinator: Sending PREPARE to all participants.")
        for participant in self.participants:
            # Simulate network delay or failure
            time.sleep(random.uniform(0.1, 0.5))
            vote = participant.prepare()
            votes.append(vote)

        # Check votes
        if all(votes):
            self.transaction_state = "committed"
            print("\nCoordinator: All participants voted YES. Sending COMMIT.")
            for participant in self.participants:
                # Simulate network delay or failure
                time.sleep(random.uniform(0.1, 0.5))
                participant.commit()
        else:
            self.transaction_state = "aborted"
            print("\nCoordinator: At least one participant voted NO or failed. Sending ABORT.")
            for participant in self.participants:
                 # Simulate network delay or failure
                time.sleep(random.uniform(0.1, 0.5))
                participant.abort()

        print(f"\nCoordinator: Transaction finished with state: {self.transaction_state}")


# Example Usage (Two-Phase Commit)
num_participants = 3
participants = [Participant(i) for i in range(num_participants)]
coordinator = Coordinator(participants)

print("--- Two-Phase Commit Simulation (Successful Commit) ---")
coordinator.start_transaction()

print("\n--- Two-Phase Commit Simulation (Aborted Transaction) ---")
# Simulate one participant being unable to commit
participants[1].set_can_commit(False)
participants[0].state = "initial" # Reset states for new transaction
participants[1].state = "initial"
participants[2].state = "initial"

coordinator.start_transaction()

# Simulate coordinator failure after prepare but before commit/abort (conceptual)
print("\n--- Two-Phase Commit Simulation (Conceptual Blocking Scenario) ---")
# In a real simulation, you'd need threads/processes and explicit failure injection.
# This demonstrates the *state* that leads to blocking.
print("Simulating coordinator sends PREPARE and all vote YES, then coordinator fails...")
participants[0].state = "initial" # Reset states for new transaction
participants[1].state = "initial"
participants[2].state = "initial"
participants[1].set_can_commit(True) # Allow participant 1 to commit this time

# Manually run prepare phase
all_ready = True
for participant in participants:
    if not participant.prepare():
        all_ready = False
        break

if all_ready:
    print("\nAll participants are in READY state.")
    print("Conceptual: Coordinator would now send COMMIT, but it fails.")
    print("Participants remain blocked in READY state, waiting for coordinator command.")
else:
     print("\nNot all participants were ready, transaction would have aborted.")


--- Two-Phase Commit Simulation (Successful Commit) ---

Coordinator: Starting transaction.
Coordinator: Sending PREPARE to all participants.
Participant 0: Received PREPARE. Voting YES.
Participant 1: Received PREPARE. Voting YES.
Participant 2: Received PREPARE. Voting YES.

Coordinator: All participants voted YES. Sending COMMIT.
Participant 0: Received COMMIT. Committing transaction.
Participant 1: Received COMMIT. Committing transaction.
Participant 2: Received COMMIT. Committing transaction.

Coordinator: Transaction finished with state: committed

--- Two-Phase Commit Simulation (Aborted Transaction) ---
Participant 1 is now set to vote NO.

Coordinator: Starting transaction.
Coordinator: Sending PREPARE to all participants.
Participant 0: Received PREPARE. Voting YES.
Participant 1: Received PREPARE. Voting NO (simulated failure/unwillingness).
Participant 2: Received PREPARE. Voting YES.

Coordinator: At least one participant voted NO or failed. Sending ABORT.
Participant 0: R

## Reservoir Sampling

### Concept Explanation
Reservoir Sampling is a family of algorithms for randomly choosing a sample of *k* items from a list (or stream) of *n* items, where *n* is a very large or unknown number, and the items are processed sequentially (one by one). The goal is to select each item with equal probability, resulting in a simple random sample of size *k*. The basic algorithm (Algorithm R) works as follows: Keep the first *k* items in memory (the "reservoir"). For each subsequent item *i* (from *k*+1 to *n*), replace an item in the reservoir with the new item with probability *k/i*. The item to replace is chosen uniformly at random from the current reservoir.

### Real-World Analogy
Imagine you want to select a random sample of 10 songs from a massive, never-ending radio stream (a stream of items with unknown size). You have a playlist (your reservoir) that can hold 10 songs (k=10). You listen to the first 10 songs and put them in your playlist. For the 11th song (i=11), you flip a coin weighted to land heads with probability 10/11. If it's heads, you randomly pick one of the 10 songs currently in your playlist and replace it with the 11th song. For the 12th song (i=12), you flip a coin weighted 10/12, and so on. This process ensures that at any point, your playlist contains a perfectly random sample of the songs you've heard so far.

### Use Cases & Trade-offs
**Use Cases:**
*   Selecting a random subset of data from a large dataset that cannot fit into memory (e.g., log files, network traffic, search results).
*   Sampling data from a streaming source where the total number of items is unknown.
*   Estimating properties of a large dataset without processing the entire dataset.
*   Selecting random users or events from a large user base or event stream.

**Trade-offs:**
*   **Advantages:**
    *   Simple and space-efficient (only needs to store the reservoir of size *k*).
    *   Can process data streams of unknown size.
    *   Guarantees that each item has an equal probability of being included in the final sample.
*   **Disadvantages:**
    *   Requires a single pass through the data.
    *   The basic algorithm requires generating random numbers for each item after the first *k*, which can be computationally intensive for very long streams. More optimized algorithms exist (e.g., Algorithm L, Algorithm X) that skip items probabilistically.
    *   The number of items to sample (*k*) must be known beforehand.



In [29]:
import random

def reservoir_sampling(stream, k):
    """
    Performs Reservoir Sampling (Algorithm R) on a data stream.

    Args:
        stream: An iterable representing the data stream (e.g., a list, generator).
        k: The desired size of the sample.

    Returns:
        A list containing the random sample of size k.
    """
    reservoir = []

    # 1. Fill the reservoir with the first k items
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            # 2. For item i (i >= k), replace a random item in the reservoir
            # with probability k/i
            j = random.randrange(0, i + 1) # randrange(0, i+1) includes i
            if j < k:
                reservoir[j] = item

    return reservoir

# Example Usage
# Simulate a large stream of numbers
large_stream = iter(range(1000)) # Imagine this is a very large or infinite stream
sample_size = 10

print(f"Sampling {sample_size} items from a stream of 1000 items...")
sample = reservoir_sampling(large_stream, sample_size)

print(f"Sampled items: {sample}")

# Verify the size of the sample
print(f"Sample size: {len(sample)}")

# Another example with different data and sample size
large_stream_2 = iter([f"item_{i}" for i in range(500)])
sample_size_2 = 5

print(f"\nSampling {sample_size_2} items from a stream of 500 items...")
sample_2 = reservoir_sampling(large_stream_2, sample_size_2)
print(f"Sampled items: {sample_2}")
print(f"Sample size: {len(sample_2)}")


Sampling 10 items from a stream of 1000 items...
Sampled items: [609, 729, 96, 641, 310, 303, 168, 999, 804, 549]
Sample size: 10

Sampling 5 items from a stream of 500 items...
Sampled items: ['item_52', 'item_147', 'item_400', 'item_215', 'item_252']
Sample size: 5


## HyperLogLog

### Concept Explanation
HyperLogLog (HLL) is a probabilistic algorithm used for estimating the number of *distinct* elements in a multiset (or stream) with high accuracy using very little memory. It's particularly useful for estimating the count of unique items in massive datasets where storing all unique items would be infeasible. HLL works by hashing each incoming element and using the properties of the hash values (specifically, the number of leading zeros) to estimate the cardinality of the set. It doesn't store the elements themselves, only a small fixed-size array of counters or "registers". The space required is logarithmic in the cardinality, allowing it to estimate cardinalities of billions with kilobytes of memory. It provides an estimate, not an exact count, and has a typical error rate of a few percent.

### Real-World Analogy
Imagine you want to estimate how many *unique* birds visit your bird feeder over a year, but you can't possibly remember or list every single bird. Instead, you decide to use a clever trick. You observe the birds and focus on a specific, rare event: a bird landing and immediately singing a song that starts with many consecutive identical notes (this is like looking for many leading zeros in a hash). If you observe a song starting with five identical notes, you know that's a much rarer event than a song starting with just one or two identical notes. The more "rare" starting patterns you observe, the more likely it is that you have a very large variety of birds visiting. HyperLogLog uses a similar statistical idea based on hash properties to estimate the number of unique items without needing to identify each one.

### Use Cases & Trade-offs
**Use Cases:**
*   Counting unique visitors to a website.
*   Estimating the number of unique search queries.
*   Counting the number of unique IP addresses connecting to a service.
*   Estimating the size of large datasets or data streams.
*   Network monitoring to count unique flows or endpoints.

**Trade-offs:**
*   **Advantages:**
    *   Extremely space-efficient, allowing for cardinality estimation on massive datasets with minimal memory usage.
    *   Can process data streams sequentially.
    *   Estimates can be merged, making it suitable for distributed environments (e.g., estimating unique users across multiple servers).
*   **Disadvantages:**
    *   Provides an estimate, not an exact count; there is always a small error rate.
    *   The accuracy is tunable but increasing accuracy requires more memory.
    *   Not suitable for applications requiring precise cardinality counts.
    *   Cannot retrieve the individual elements, only the estimated count of unique elements.


In [31]:
import hashlib
import math
import random

class HyperLogLog:
    def __init__(self, num_registers_log2):
        """
        Initializes the HyperLogLog structure.

        Args:
            num_registers_log2: Log base 2 of the number of registers (m).
        """
        self.m = 1 << num_registers_log2 # Number of registers (2^num_registers_log2)
        self.registers = [0] * self.m
        self.alpha_m = self._get_alpha(self.m)

    def _hash(self, item):
        """Computes a hash for the item."""
        # Using SHA-256 for demonstration, but a non-cryptographic hash is often faster
        # and sufficient for HLL (e.g., MurmurHash, FNV).
        # We need enough bits to find leading zeros. SHA-256 is 256 bits.
        return hashlib.sha256(str(item).encode('utf-8')).digest()

    def _get_alpha(self, m):
        """Calculates the bias correction factor alpha_m."""
        if m == 16:
            return 0.673
        elif m == 32:
            return 0.697
        elif m == 64:
            return 0.709
        else:
            return 0.7213 / (1 + 1.079 / m) # Formula for m >= 128

    def _get_leading_zeros(self, hash_bytes):
        """Counts the number of leading zero bits in the hash."""
        count = 0
        for byte in hash_bytes:
            if byte == 0:
                count += 8
            else:
                count += bin(byte)[2:].zfill(8).find('1')
                break
        return count

    def add(self, item):
        """Adds an item to the HyperLogLog structure."""
        hashed_item = self._hash(item)
        # Use the first log2(m) bits of the hash to determine the register index
        # Assuming SHA-256 returns bytes
        # Convert first few bytes to an integer for index
        index_bytes_len = (self.m - 1).bit_length() // 8 + (1 if (self.m - 1).bit_length() % 8 > 0 else 0)
        if index_bytes_len == 0: index_bytes_len = 1 # Handle m=1 or m=2 cases
        index_int = int.from_bytes(hashed_item[:index_bytes_len], 'big')
        register_index = index_int % self.m

        # Count leading zeros in the *rest* of the hash
        # We need to make sure we use enough bits from the hash for estimating cardinality
        # A common approach is to use the first bits for the register index, and the rest for the rho calculation
        # For simplicity here, we'll use the full hash but be mindful of the index bits
        # A more precise implementation would use the bits *after* the index bits for rho
        # Let's use the standard approach: index from first bits, rho from remaining bits.

        # Simplified: Use index_int for index, and the full hash for rho calculation
        # This is not strictly correct per the original paper but is simpler for simulation.
        # A better approach would involve bit manipulation to get the bits after the index.
        # For this simulation, let's find the first '1' bit across the *entire* hash.
        # This is simpler to implement than extracting bits after the index bits.
        # The position of the first '1' bit (rho) is used for the register value.
        # rho(x) = position of the least significant bit of x, plus one. OR
        # rho(x) = position of the first *set* bit from the left (most significant), plus one.
        # Let's use the latter (position of first '1' from MSB).

        # A standard HLL uses the position of the first *unset* bit (0) or first *set* bit (1) after the index bits.
        # Let's use the position of the first '1' bit starting from the beginning of the hash bytes.
        # The number of leading zeros is equivalent to the position of the first '1' bit minus one.
        # rho = number of leading zeros + 1
        rho = 1 # Position of the first bit is 1
        for byte in hashed_item:
            if byte == 0:
                rho += 8
            else:
                # Find position of first '1' in this non-zero byte
                rho += bin(byte)[2:].zfill(8).find('1')
                break

        self.registers[register_index] = max(self.registers[register_index], rho)


    def estimate_cardinality(self):
        """Estimates the cardinality (number of distinct elements)."""
        # Calculate the harmonic mean of the register values
        harmonic_mean_reciprocal = sum(2 ** (-register_value) for register_value in self.registers)
        if harmonic_mean_reciprocal == 0: # Avoid division by zero
            return 0

        harmonic_mean = self.m / harmonic_mean_reciprocal

        # Apply the bias correction factor
        raw_estimate = self.alpha_m * (self.m ** 2) / harmonic_mean

        # Apply corrections for small and large cardinalities (simplified)
        # Small range correction: if estimate is small, use linear counting
        if raw_estimate <= 2.5 * self.m:
            V = self.registers.count(0) # Number of zero registers
            if V > 0:
                linear_counting_estimate = self.m * math.log(self.m / V)
                return linear_counting_estimate
            else:
                return raw_estimate # No zeros, use raw estimate

        # Large range correction (not strictly necessary for basic simulation,
        # but in full HLL, adjust if estimate exceeds 2^32/3 or 2^64/3)
        # We'll skip this for simplicity in the simulation.

        return raw_estimate

# Example Usage
# num_registers_log2 = 4 means m = 2^4 = 16 registers
# num_registers_log2 = 6 means m = 2^6 = 64 registers
# num_registers_log2 = 10 means m = 2^10 = 1024 registers (more accurate)

# Let's use 1024 registers for a reasonable estimate
hll = HyperLogLog(num_registers_log2=10)
print(f"Initialized HyperLogLog with {hll.m} registers.")

# Add some elements (some duplicates)
elements = [1, 2, 3, 1, 4, 5, 6, 2, 7, 8, 9, 3, 10, 11, 12, 4, 13, 14, 15, 16]
distinct_elements = set(elements)
actual_cardinality = len(distinct_elements)

print(f"\nAdding {len(elements)} elements to HLL (Actual distinct: {actual_cardinality})...")
for element in elements:
    hll.add(element)

estimated_cardinality = hll.estimate_cardinality()
print(f"Estimated cardinality: {estimated_cardinality:.2f}")
print(f"Estimation error: {abs(estimated_cardinality - actual_cardinality) / actual_cardinality * 100:.2f}%")


# Add many more elements
more_elements = list(range(1000)) + list(range(500, 1500)) # Many duplicates
distinct_more_elements = set(more_elements)
actual_cardinality_2 = len(distinct_more_elements)

hll_2 = HyperLogLog(num_registers_log2=10) # New HLL instance
print(f"\nAdding {len(more_elements)} elements to a new HLL (Actual distinct: {actual_cardinality_2})...")
for element in more_elements:
    hll_2.add(element)

estimated_cardinality_2 = hll_2.estimate_cardinality()
print(f"Estimated cardinality: {estimated_cardinality_2:.2f}")
print(f"Estimation error: {abs(estimated_cardinality_2 - actual_cardinality_2) / actual_cardinality_2 * 100:.2f}%")


Initialized HyperLogLog with 1024 registers.

Adding 20 elements to HLL (Actual distinct: 16)...
Estimated cardinality: 747471.69
Estimation error: 4671598.06%

Adding 2000 elements to a new HLL (Actual distinct: 1500)...
Estimated cardinality: 319479.96
Estimation error: 21198.66%


## CRDTs

### Concept Explanation
CRDTs (Conflict-free Replicated Data Types) are data structures designed to be replicated across multiple servers or nodes, where updates can happen concurrently and asynchronously without coordination. The key property of CRDTs is that they are mathematically guaranteed to converge to the same state on all replicas, regardless of the order in which concurrent updates are applied. This "strong eventual consistency" is achieved by designing the data type and its operations such that they are commutative, associative, and idempotent, or by using a merge function that has these properties. There are two main types:
*   **Operation-based CRDTs:** Replicas exchange the operations that modify the data. Operations must be delivered reliably and in a way that allows them to be applied in any order (often requires causal ordering).
*   **State-based CRDTs:** Replicas exchange their full local state. The state includes enough information to allow any two states to be merged deterministically using a join function (e.g., union, maximum).

### Real-World Analogy
Imagine a shared shopping list application used by multiple people on their phones. Everyone can add items to their local list even when offline. When they go online, their lists sync.

*   **Operation-based:** When you add "milk" to your list, your phone sends a message "Add 'milk'". Other phones receive this message and add "milk" to their lists. The order of receiving "Add 'milk'" and "Add 'bread'" messages doesn't matter; both items will end up on the final list.
*   **State-based:** When you add "milk", your phone updates its entire list to include "milk". When syncing, your phone sends its whole list to others. Others receive your list and merge it with theirs by taking the union of all items. If one person added "milk" and another added "bread" concurrently, merging their lists (e.g., {"milk"} union {"bread"}) results in {"milk", "bread"} on both sides, regardless of whose list arrived first.

CRDTs are like designing the shopping list app so that no matter who adds what or in what order they sync, everyone eventually ends up with the same final list of items, without needing a central server to decide the "correct" list.

### Use Cases & Trade-offs
**Use Cases:**
*   Collaborative editing applications (like Google Docs or Figma) where multiple users modify a document concurrently.
*   Shared data structures in distributed databases and key-value stores (e.g., Redis, Riak).
*   Multiplayer online games for synchronizing game state.
*   Offline-first mobile applications where users can make changes offline and sync later.
*   Implementing features like shared counters, sets, lists, and registers in distributed systems.

**Trade-offs:**
*   **Advantages:**
    *   Provides strong eventual consistency without requiring complex distributed transactions or consensus protocols for every update.
    *   High availability and partition tolerance; updates can proceed even during network partitions.
    *   Simplifies development of collaborative or eventually consistent features.
*   **Disadvantages:**
    *   Can consume more memory than traditional data structures (especially state-based CRDTs, which may need to store metadata for convergence).
    *   Designing and implementing correct CRDTs can be challenging for complex data types.
    *   Operation-based CRDTs may require causal message delivery.
    *   May accumulate "garbage" (e.g., tombstones for removed elements) over time, requiring compaction.


In [33]:
class GCounter:
    """
    A Grow-only Counter (GCounter) CRDT.
    Increments are only additions, and the value only increases.
    Merging is element-wise maximum.
    """
    def __init__(self, node_id, num_nodes):
        self.node_id = node_id
        self.num_nodes = num_nodes
        # Vector to store increments from each node
        self.counts = [0] * num_nodes

    def increment(self, amount=1):
        """Increments the counter at this node."""
        if amount > 0:
            self.counts[self.node_id] += amount
            print(f"Node {self.node_id}: Incremented by {amount}. State: {self.counts}")

    def value(self):
        """Returns the total value of the counter."""
        return sum(self.counts)

    def merge(self, other_counter):
        """Merges this counter's state with another counter's state."""
        if len(self.counts) != len(other_counter.counts):
            raise ValueError("Cannot merge GCounter with different number of nodes.")

        print(f"Node {self.node_id}: Merging with state {other_counter.counts}")
        for i in range(self.num_nodes):
            self.counts[i] = max(self.counts[i], other_counter.counts[i])
        print(f"Node {self.node_id}: Merged state: {self.counts}")

    def get_state(self):
        """Returns the current state (the counts vector)."""
        return list(self.counts)

# Example Usage: Simulating a Grow-only Counter across replicas
num_replicas = 3
replica_0 = GCounter(node_id=0, num_nodes=num_replicas)
replica_1 = GCounter(node_id=1, num_nodes=num_replicas)
replica_2 = GCounter(node_id=2, num_nodes=num_replicas)

replicas = [replica_0, replica_1, replica_2]

print("--- GCounter CRDT Simulation ---")

# Simulate concurrent increments on different replicas
print("\nSimulating concurrent increments:")
replica_0.increment() # R0: [1, 0, 0]
replica_1.increment(2) # R1: [0, 2, 0]
replica_2.increment() # R2: [0, 0, 1]
replica_0.increment() # R0: [2, 0, 0]

print("\nStates after concurrent increments:")
for i, rep in enumerate(replicas):
    print(f"Replica {i} state: {rep.get_state()}")
    print(f"Replica {i} value: {rep.value()}")

# Simulate merging states between replicas
print("\nSimulating merging states:")

# Merge R1 into R0
print("\nMerging Replica 1 into Replica 0:")
replica_0.merge(replica_1) # R0 becomes [max(2,0), max(0,2), max(0,0)] = [2, 2, 0]

# Merge R2 into R0
print("\nMerging Replica 2 into Replica 0:")
replica_0.merge(replica_2) # R0 becomes [max(2,0), max(2,0), max(0,1)] = [2, 2, 1]

# Merge R0 (updated) into R1
print("\nMerging Replica 0 into Replica 1:")
replica_1.merge(replica_0) # R1 becomes [max(0,2), max(2,2), max(0,1)] = [2, 2, 1]

# Merge R0 (updated) into R2
print("\nMerging Replica 0 into Replica 2:")
replica_2.merge(replica_0) # R2 becomes [max(0,2), max(0,2), max(1,1)] = [2, 2, 1]


print("\nStates after merging:")
for i, rep in enumerate(replicas):
    print(f"Replica {i} state: {rep.get_state()}")
    print(f"Replica {i} value: {rep.value()}")

# Verify convergence: all replicas should have the same state and value
all_ready = all(rep.get_state() == [2, 2, 1] for rep in replicas)
print(f"\nHave all replicas converged to the same state ([2, 2, 1])? {all_ready}")
print(f"Final converged value: {replica_0.value()}") # Value should be 2+2+1 = 5

--- GCounter CRDT Simulation ---

Simulating concurrent increments:
Node 0: Incremented by 1. State: [1, 0, 0]
Node 1: Incremented by 2. State: [0, 2, 0]
Node 2: Incremented by 1. State: [0, 0, 1]
Node 0: Incremented by 1. State: [2, 0, 0]

States after concurrent increments:
Replica 0 state: [2, 0, 0]
Replica 0 value: 2
Replica 1 state: [0, 2, 0]
Replica 1 value: 2
Replica 2 state: [0, 0, 1]
Replica 2 value: 1

Simulating merging states:

Merging Replica 1 into Replica 0:
Node 0: Merging with state [0, 2, 0]
Node 0: Merged state: [2, 2, 0]

Merging Replica 2 into Replica 0:
Node 0: Merging with state [0, 0, 1]
Node 0: Merged state: [2, 2, 1]

Merging Replica 0 into Replica 1:
Node 1: Merging with state [2, 2, 1]
Node 1: Merged state: [2, 2, 1]

Merging Replica 0 into Replica 2:
Node 2: Merging with state [2, 2, 1]
Node 2: Merged state: [2, 2, 1]

States after merging:
Replica 0 state: [2, 2, 1]
Replica 0 value: 5
Replica 1 state: [2, 2, 1]
Replica 1 value: 5
Replica 2 state: [2, 2, 1]

## Sharding Algorithms

### Concept Explanation
Sharding is a database partitioning technique that divides a large dataset into smaller, more manageable pieces called shards. Each shard is an independent database or data store, typically hosted on a separate server. This distribution of data across multiple machines allows databases to scale horizontally, handling larger volumes of data and higher transaction loads than a single server could manage. Sharding algorithms determine how data is distributed among the shards, often based on a shard key derived from the data itself (e.g., user ID, geographical location).

### Real-World Analogy
Imagine a massive library with millions of books. Instead of keeping all books in one giant building, which would be difficult to manage and search, the library is divided into several smaller, independent branches (shards) across the city. Each branch contains a portion of the total collection, perhaps organized alphabetically by author or by genre (the shard key). When you want a book, you go to the specific branch where that book is located. This makes it easier to manage the collection and allows more people to access different books simultaneously.

### Use Cases & Trade-offs
**Use Cases:**
*   Scaling large databases that exceed the capacity of a single server.
*   Improving performance by distributing query load across multiple servers.
*   Reducing the amount of data that needs to be scanned for a query (if the query uses the shard key).
*   Increasing availability; if one shard fails, other shards can remain operational.

**Trade-offs:**
*   **Advantages:**
    *   Enables horizontal scalability for large datasets and high traffic.
    *   Can improve read and write performance by distributing load.
    *   Increases availability (though managing distributed transactions across shards is complex).
*   **Disadvantages:**
    *   Adds complexity to the system architecture (application logic needs to know which shard to access).
    *   Implementing joins and queries across multiple shards can be challenging.
    *   Resharding (redistributing data when adding or removing shards) can be a complex and potentially disruptive operation.
    *   Choosing the right shard key is critical for even data distribution and performance; a poor shard key can lead to hotspots (uneven load on shards).


In [35]:
import hashlib

def simple_hash_sharding(key, num_shards):
    """
    Distributes keys to shards using a simple hash function.

    Args:
        key: The shard key (e.g., user ID, item ID).
        num_shards: The total number of available shards.

    Returns:
        The index of the shard (0 to num_shards-1).
    """
    # Use SHA-256 hash and take the modulo of the number of shards
    # This provides a relatively even distribution for numerical shard keys
    # For non-numerical keys, ensure proper encoding
    hash_value = int(hashlib.sha256(str(key).encode('utf-8')).hexdigest(), 16)
    shard_index = hash_value % num_shards
    return shard_index

# Example Usage
num_shards = 4
print(f"Simulating Sharding with {num_shards} shards.")

# Simulate mapping some keys to shards
keys = [101, 205, 310, 455, 500, 1001, 1002, 2005, 3010, 4055]

print("\nMapping keys to shards:")
for key in keys:
    shard = simple_hash_sharding(key, num_shards)
    print(f"Key {key} maps to Shard {shard}")

# Demonstrate data distribution (conceptual)
shard_data = {i: [] for i in range(num_shards)}
for key in keys:
    shard = simple_hash_sharding(key, num_shards)
    shard_data[shard].append(key)

print("\nConceptual distribution of keys across shards:")
for shard, data in shard_data.items():
    print(f"Shard {shard}: {data}")

# Simulate adding more shards (requires re-sharding in a real system)
# This simple hash sharding doesn't handle adding/removing shards gracefully
# Consistent Hashing (covered earlier) is better for this.
new_num_shards = 5
print(f"\nSimulating adding a shard (now {new_num_shards} shards) - Requires Re-sharding:")

new_shard_data = {i: [] for i in range(new_num_shards)}
for key in keys:
    # Data needs to be re-mapped to the new number of shards
    new_shard = simple_hash_sharding(key, new_num_shards)
    new_shard_data[new_shard].append(key)

print("\nConceptual distribution of keys after re-sharding:")
for shard, data in new_shard_data.items():
    print(f"Shard {shard}: {data}")

Simulating Sharding with 4 shards.

Mapping keys to shards:
Key 101 maps to Shard 0
Key 205 maps to Shard 0
Key 310 maps to Shard 0
Key 455 maps to Shard 3
Key 500 maps to Shard 2
Key 1001 maps to Shard 1
Key 1002 maps to Shard 1
Key 2005 maps to Shard 0
Key 3010 maps to Shard 1
Key 4055 maps to Shard 0

Conceptual distribution of keys across shards:
Shard 0: [101, 205, 310, 2005, 4055]
Shard 1: [1001, 1002, 3010]
Shard 2: [500]
Shard 3: [455]

Simulating adding a shard (now 5 shards) - Requires Re-sharding:

Conceptual distribution of keys after re-sharding:
Shard 0: [1001, 1002]
Shard 1: []
Shard 2: [101, 205, 3010, 4055]
Shard 3: [310, 455]
Shard 4: [500, 2005]


## MapReduce

### Concept Explanation
MapReduce is a programming model and an associated implementation for processing and generating large datasets with a parallel, distributed algorithm on a cluster. It consists of two main phases:

1.  **Map:** The input data is split into independent chunks. The Map function processes each chunk and produces a set of intermediate key-value pairs. This phase is typically performed in parallel across many worker nodes.
2.  **Reduce:** The intermediate key-value pairs generated by the Map phase are grouped by key. The Reduce function processes the values for each key and produces a final output. This phase is also performed in parallel, often on different worker nodes than the Map phase.

A framework (like Apache Hadoop MapReduce or Apache Spark) handles the complexities of distributing the data and computation, managing failures, and coordinating the Map and Reduce tasks.

### Real-World Analogy
Imagine you have a massive pile of documents (input data) and you want to count the occurrences of each word.

*   **Map:** You distribute the documents among several assistants (Map workers). Each assistant reads their documents, identifies each word, and writes down the word and the number '1' next to it on a slip of paper (intermediate key-value pairs like (word, 1)).
*   **Shuffle and Sort (handled by the framework):** The slips of paper from all assistants are collected and sorted so that all slips for the same word are grouped together.
*   **Reduce:** You assign groups of slips for the same word to other assistants (Reduce workers). Each assistant receives all slips for their assigned word and sums up the numbers on the slips (processes the values for each key). They write the word and the total count on a final report (final output).

This parallel process allows you to count words in a huge collection of documents much faster than one person doing it alone.

### Use Cases & Trade-offs
**Use Cases:**
*   Processing large log files to extract information (e.g., counting errors, analyzing user behavior).
*   Building search indexes by processing web pages or documents.
*   Analyzing large datasets for reporting and analytics.
*   Performing ETL (Extract, Transform, Load) operations on big data.
*   Graph processing and analysis.

**Trade-offs:**
*   **Advantages:**
    *   Highly scalable; can process petabytes of data on large clusters.
    *   Fault-tolerant; the framework can handle worker failures and retry tasks.
    *   Simplifies parallel programming for certain types of batch processing tasks.
*   **Disadvantages:**
    *   Primarily designed for batch processing; not suitable for real-time or interactive queries.
    *   Can be less efficient for iterative algorithms or tasks that require sharing mutable state between tasks.
    *   The programming model (Map and Reduce functions) can be restrictive for complex data processing workflows.
    *   Can have high latency due to the batch-oriented nature and the overhead of reading/writing intermediate data to disk (common in older implementations like Hadoop MapReduce).


In [37]:
from collections import defaultdict

def map_function(text_chunk):
    """Simulates the Map phase for word counting."""
    intermediate_pairs = []
    # Simple tokenization: split by whitespace and convert to lowercase
    words = text_chunk.lower().split()
    for word in words:
        # In a real MapReduce, this would emit (word, 1)
        intermediate_pairs.append((word, 1))
    return intermediate_pairs

def reduce_function(key, values):
    """Simulates the Reduce phase for word counting."""
    # Sum up the counts for each word
    final_count = sum(values)
    # In a real MapReduce, this would emit (word, total_count)
    return (key, final_count)

def simulate_mapreduce(input_data, num_map_workers, num_reduce_workers):
    """Basic simulation of the MapReduce process."""
    print("--- MapReduce Simulation (Word Count) ---")

    # 1. Splitting the input data (conceptual)
    # In a real system, this is handled by the framework and HDFS/storage
    chunk_size = len(input_data) // num_map_workers + (len(input_data) % num_map_workers > 0)
    data_chunks = [input_data[i:i + chunk_size] for i in range(0, len(input_data), chunk_size)]
    print(f"Input data split into {len(data_chunks)} chunks for {num_map_workers} map workers.")

    # 2. Map Phase
    print("\nStarting Map Phase...")
    all_intermediate_pairs = []
    for i, chunk in enumerate(data_chunks):
        # Simulate running map function on a worker
        print(f"Map Worker {i+1} processing chunk...")
        intermediate_pairs = map_function(chunk)
        all_intermediate_pairs.extend(intermediate_pairs)
        # In a real system, intermediate results are typically written to local disk

    print(f"\nMap Phase finished. Generated {len(all_intermediate_pairs)} intermediate pairs.")
    # print(f"Intermediate pairs (sample): {all_intermediate_pairs[:20]}...") # Avoid printing too much output


    # 3. Shuffle and Sort Phase (handled by framework - conceptual)
    # Group intermediate pairs by key
    print("\nStarting Shuffle and Sort Phase (Grouping by key)...")
    grouped_intermediate_data = defaultdict(list)
    for key, value in all_intermediate_pairs:
        grouped_intermediate_data[key].append(value)
    print(f"Intermediate data grouped for {len(grouped_intermediate_data)} unique keys.")

    # 4. Reduce Phase
    print("\nStarting Reduce Phase...")
    final_output = []
    # In a real system, groups of data for each key are sent to reduce workers
    # We'll just iterate through the grouped data here
    items_to_reduce = list(grouped_intermediate_data.items())
    reduce_chunk_size = len(items_to_reduce) // num_reduce_workers + (len(items_to_reduce) % num_reduce_workers > 0)
    reduce_chunks = [items_to_reduce[i:i + reduce_chunk_size] for i in range(0, len(items_to_reduce), reduce_chunk_size)]

    for i, chunk in enumerate(reduce_chunks):
        print(f"Reduce Worker {i+1} processing {len(chunk)} keys...")
        for key, values in chunk:
            final_result = reduce_function(key, values)
            final_output.append(final_result)
        # In a real system, final results are written to HDFS/storage

    print("\nReduce Phase finished.")

    # Sort final output for readability
    final_output.sort()

    print("\nFinal Word Counts:")
    # print up to 20 words for brevity
    for word, count in final_output[:20]:
        print(f"{word}: {count}")
    if len(final_output) > 20:
        print("...")

# Example Input Data
input_text = """
MapReduce is a programming model and an associated implementation for processing and generating large datasets
with a parallel, distributed algorithm on a cluster.
It consists of two main phases: Map and Reduce.
The Map phase processes input data and produces intermediate key-value pairs.
The Reduce phase processes the intermediate pairs and produces the final output.
MapReduce is great for batch processing but not for real-time queries.
"""

simulate_mapreduce(input_text.split(), num_map_workers=3, num_reduce_workers=2)

# Another example with different data
input_numbers = [1, 2, 3, 1, 4, 5, 2, 3, 1, 6, 7, 8, 2, 9, 1]

def map_number_count(number):
    return (number, 1)

def reduce_number_count(key, values):
    return (key, sum(values))

print("\n--- MapReduce Simulation (Number Count) ---")

# Simulate mapping
intermediate_numbers = []
for number in input_numbers:
    intermediate_numbers.append(map_number_count(number))

# Simulate shuffling/grouping
grouped_numbers = defaultdict(list)
for key, value in intermediate_numbers:
    grouped_numbers[key].append(value)

# Simulate reducing
final_number_counts = []
for key, values in grouped_numbers.items():
    final_number_counts.append(reduce_number_count(key, values))

final_number_counts.sort()

print("\nFinal Number Counts:")
for number, count in final_number_counts:
    print(f"{number}: {count}")


--- MapReduce Simulation (Word Count) ---
Input data split into 3 chunks for 3 map workers.

Starting Map Phase...
Map Worker 1 processing chunk...


AttributeError: 'list' object has no attribute 'lower'

In [38]:
from collections import defaultdict

def map_function(text_chunk):
    """Simulates the Map phase for word counting."""
    intermediate_pairs = []
    # Simple tokenization: split by whitespace and convert to lowercase
    # Fix: text_chunk is a string, so .lower() and .split() are correct here.
    words = text_chunk.lower().split()
    for word in words:
        # In a real MapReduce, this would emit (word, 1)
        intermediate_pairs.append((word, 1))
    return intermediate_pairs

def reduce_function(key, values):
    """Simulates the Reduce phase for word counting."""
    # Sum up the counts for each word
    final_count = sum(values)
    # In a real MapReduce, this would emit (word, total_count)
    return (key, final_count)

def simulate_mapreduce(input_data, num_map_workers, num_reduce_workers):
    """Basic simulation of the MapReduce process."""
    print("--- MapReduce Simulation (Word Count) ---")

    # 1. Splitting the input data (conceptual)
    # In a real system, this is handled by the framework and HDFS/storage
    # Fix: Pass the raw input_data string to split into chunks, not already split words
    data_length = len(input_data)
    chunk_size = data_length // num_map_workers + (data_length % num_map_workers > 0)
    # Split the input string into chunks
    data_chunks = [input_data[i:i + chunk_size] for i in range(0, data_length, chunk_size)]

    print(f"Input data split into {len(data_chunks)} chunks for {num_map_workers} map workers.")

    # 2. Map Phase
    print("\nStarting Map Phase...")
    all_intermediate_pairs = []
    for i, chunk in enumerate(data_chunks):
        # Simulate running map function on a worker
        print(f"Map Worker {i+1} processing chunk...")
        intermediate_pairs = map_function(chunk)
        all_intermediate_pairs.extend(intermediate_pairs)
        # In a real system, intermediate results are typically written to local disk

    print(f"\nMap Phase finished. Generated {len(all_intermediate_pairs)} intermediate pairs.")
    # print(f"Intermediate pairs (sample): {all_intermediate_pairs[:20]}...") # Avoid printing too much output


    # 3. Shuffle and Sort Phase (handled by framework - conceptual)
    # Group intermediate pairs by key
    print("\nStarting Shuffle and Sort Phase (Grouping by key)...")
    grouped_intermediate_data = defaultdict(list)
    for key, value in all_intermediate_pairs:
        grouped_intermediate_data[key].append(value)
    print(f"Intermediate data grouped for {len(grouped_intermediate_data)} unique keys.")

    # 4. Reduce Phase
    print("\nStarting Reduce Phase...")
    final_output = []
    # In a real system, groups of data for each key are sent to reduce workers
    # We'll just iterate through the grouped data here
    items_to_reduce = list(grouped_intermediate_data.items())
    reduce_chunk_size = len(items_to_reduce) // num_reduce_workers + (len(items_to_reduce) % num_reduce_workers > 0)
    reduce_chunks = [items_to_reduce[i:i + reduce_chunk_size] for i in range(0, len(items_to_reduce), reduce_chunk_size)]

    for i, chunk in enumerate(reduce_chunks):
        print(f"Reduce Worker {i+1} processing {len(chunk)} keys...")
        for key, values in chunk:
            final_result = reduce_function(key, values)
            final_output.append(final_result)
        # In a real system, final results are written to HDFS/storage

    print("\nReduce Phase finished.")

    # Sort final output for readability
    final_output.sort()

    print("\nFinal Word Counts:")
    # print up to 20 words for brevity
    for word, count in final_output[:20]:
        print(f"{word}: {count}")
    if len(final_output) > 20:
        print("...")

# Example Input Data
input_text = """
MapReduce is a programming model and an associated implementation for processing and generating large datasets
with a parallel, distributed algorithm on a cluster.
It consists of two main phases: Map and Reduce.
The Map phase processes input data and produces intermediate key-value pairs.
The Reduce phase processes the intermediate pairs and produces the final output.
MapReduce is great for batch processing but not for real-time queries.
"""

# Fix: Pass the raw input_text string, not the split list
simulate_mapreduce(input_text, num_map_workers=3, num_reduce_workers=2)

# Another example with different data
input_numbers = [1, 2, 3, 1, 4, 5, 2, 3, 1, 6, 7, 8, 2, 9, 1]

def map_number_count(number):
    return (number, 1)

def reduce_number_count(key, values):
    return (key, sum(values))

print("\n--- MapReduce Simulation (Number Count) ---")

# Simulate mapping
intermediate_numbers = []
for number in input_numbers:
    intermediate_numbers.append(map_number_count(number))

# Simulate shuffling/grouping
grouped_numbers = defaultdict(list)
for key, value in intermediate_numbers:
    grouped_numbers[key].append(value)

# Simulate reducing
final_number_counts = []
for key, values in grouped_numbers.items():
    final_number_counts.append(reduce_number_count(key, values))

final_number_counts.sort()

print("\nFinal Number Counts:")
for number, count in final_number_counts:
    print(f"{number}: {count}")

--- MapReduce Simulation (Word Count) ---
Input data split into 3 chunks for 3 map workers.

Starting Map Phase...
Map Worker 1 processing chunk...
Map Worker 2 processing chunk...
Map Worker 3 processing chunk...

Map Phase finished. Generated 68 intermediate pairs.

Starting Shuffle and Sort Phase (Grouping by key)...
Intermediate data grouped for 49 unique keys.

Starting Reduce Phase...
Reduce Worker 1 processing 25 keys...
Reduce Worker 2 processing 24 keys...

Reduce Phase finished.

Final Word Counts:
a: 3
algorit: 1
an: 1
and: 5
associated: 1
batch: 1
but: 1
cluster.: 1
consists: 1
data: 1
datasets: 1
distributed: 1
educe: 1
final: 1
for: 3
generating: 1
great: 1
hm: 1
implementation: 1
input: 1
...

--- MapReduce Simulation (Number Count) ---

Final Number Counts:
1: 4
2: 3
3: 2
4: 1
5: 1
6: 1
7: 1
8: 1
9: 1


## Tail Latency Reduction (Hedged Requests)

### Concept Explanation
Tail latency refers to the latency experienced by the slowest requests in a system. High tail latency can significantly impact user experience, especially in systems where a single user request fans out to many backend services. Hedged requests (also known as speculative execution) are a technique to reduce tail latency. When a request is sent to a server (or service), a backup or "hedged" request is sent to another server (or a different replica of the same service) if the original request does not complete within a certain timeout. The system then uses the result from whichever request completes first and cancels the other.

### Real-World Analogy
Imagine you are ordering a popular item from a busy online store. You place your order (the original request). If the website doesn't confirm your order within a few seconds (a timeout), you quickly try ordering the same item again on a different device or through a different browser session (the hedged request). Whichever order confirmation arrives first is the one you proceed with, and you cancel the other one. This increases your chance of getting a quick confirmation compared to just waiting indefinitely for the first attempt.

### Use Cases & Trade-offs
**Use Cases:**
*   Improving the responsiveness of systems where user requests depend on multiple backend calls (e.g., search engines, recommendation systems).
*   Reducing the impact of slow or overloaded servers in a distributed system.
*   Optimizing read operations in replicated data stores.

**Trade-offs:**
*   **Advantages:**
    *   Significantly reduces tail latency, leading to a better user experience.
    *   Improves overall system throughput by reducing the time spent waiting for slow requests.
*   **Disadvantages:**
    *   Increases resource utilization (CPU, network) by sending duplicate requests.
    *   Can potentially overload backend servers if not implemented carefully (e.g., if the timeout is too short or too many requests are hedged).
    *   Requires logic to handle duplicate responses and cancel outstanding requests.
    *   May not be suitable for write operations unless they are idempotent (can be safely executed multiple times).


In [40]:
import time
import random
import threading

class MockServer:
    """Simulates a server with variable response times."""
    def __init__(self, id):
        self.id = id

    def process_request(self, request_id, processing_time):
        """Simulates processing a request with a delay."""
        # print(f"Server {self.id}: Starting processing for request {request_id} (takes {processing_time:.2f}s)")
        time.sleep(processing_time)
        # print(f"Server {self.id}: Finished processing for request {request_id}")
        return f"Response from Server {self.id} for request {request_id}"

class Client:
    """Simulates a client sending requests with hedging."""
    def __init__(self, servers):
        self.servers = servers
        self._lock = threading.Lock()
        self._completed_requests = {} # To store results of completed requests

    def send_request(self, request_id, primary_server, hedged_server, timeout, primary_processing_time, hedged_processing_time):
        """Sends a primary request and potentially a hedged request."""
        print(f"\nClient: Sending primary request {request_id} to Server {primary_server.id}")
        start_time = time.time()
        result = None
        completed = threading.Event() # Event to signal completion

        def primary_task():
            res = primary_server.process_request(request_id, primary_processing_time)
            with self._lock:
                if request_id not in self._completed_requests:
                    self._completed_requests[request_id] = (res, time.time() - start_time, primary_server.id)
                    print(f"Client: Request {request_id} completed by Server {primary_server.id} in {self._completed_requests[request_id][1]:.2f}s")
                    completed.set() # Signal completion

        def hedged_task():
            # Wait for timeout or primary completion
            completed.wait(timeout)
            with self._lock:
                # Only send hedged request if not already completed
                if request_id not in self._completed_requests:
                    print(f"Client: Primary request {request_id} timed out after {timeout}s. Sending hedged request to Server {hedged_server.id}")
                    res = hedged_server.process_request(request_id, hedged_processing_time)
                    if request_id not in self._completed_requests: # Check again in case primary finished while hedged was processing
                         self._completed_requests[request_id] = (res, time.time() - start_time, hedged_server.id)
                         print(f"Client: Hedged request {request_id} completed by Server {hedged_server.id} in {self._completed_requests[request_id][1]:.2f}s")
                         completed.set() # Signal completion
                    else:
                        # Hedged finished, but primary also just finished. Use primary's result.
                        print(f"Client: Hedged request {request_id} finished, but primary already completed.")


        primary_thread = threading.Thread(target=primary_task)
        hedged_thread = threading.Thread(target=hedged_task)

        primary_thread.start()
        hedged_thread.start()

        # Wait for the request to complete (either primary or hedged)
        completed.wait()

        with self._lock:
            return self._completed_requests.get(request_id)


# Example Usage
# Simulate a few servers with varying processing times
servers = [MockServer(0), MockServer(1), MockServer(2)]
client = Client(servers)

# Simulate a request where the primary server is slow
print("--- Hedged Request Simulation ---")
request_id_1 = 1
primary_server_1 = servers[0]
hedged_server_1 = servers[1]
timeout_1 = 0.5 # Send hedged request if primary takes longer than 0.5s
primary_processing_time_1 = 1.0 # Primary is slow
hedged_processing_time_1 = 0.2 # Hedged is fast

result_1, latency_1, server_id_1 = client.send_request(request_id_1, primary_server_1, hedged_server_1, timeout_1, primary_processing_time_1, hedged_processing_time_1)
print(f"Result for request {request_id_1}: {result_1}, Latency: {latency_1:.2f}s (from Server {server_id_1})")


# Simulate a request where the primary server is fast
request_id_2 = 2
primary_server_2 = servers[2]
hedged_server_2 = servers[0]
timeout_2 = 0.5
primary_processing_time_2 = 0.2 # Primary is fast
hedged_processing_time_2 = 1.0 # Hedged is slow

result_2, latency_2, server_id_2 = client.send_request(request_id_2, primary_server_2, hedged_server_2, timeout_2, primary_processing_time_2, hedged_processing_time_2)
print(f"Result for request {request_id_2}: {result_2}, Latency: {latency_2:.2f}s (from Server {server_id_2})")

# Simulate a request where both are fast (no hedging needed)
request_id_3 = 3
primary_server_3 = servers[1]
hedged_server_3 = servers[2]
timeout_3 = 0.5
primary_processing_time_3 = 0.3
hedged_processing_time_3 = 0.4

result_3, latency_3, server_id_3 = client.send_request(request_id_3, primary_server_3, hedged_server_3, timeout_3, primary_processing_time_3, hedged_processing_time_3)
print(f"Result for request {request_id_3}: {result_3}, Latency: {latency_3:.2f}s (from Server {server_id_3})")



--- Hedged Request Simulation ---

Client: Sending primary request 1 to Server 0
Client: Primary request 1 timed out after 0.5s. Sending hedged request to Server 1
Client: Hedged request 1 completed by Server 1 in 0.70s
Result for request 1: Response from Server 1 for request 1, Latency: 0.70s (from Server 1)

Client: Sending primary request 2 to Server 2
Client: Request 2 completed by Server 2 in 0.20s
Result for request 2: Response from Server 2 for request 2, Latency: 0.20s (from Server 2)

Client: Sending primary request 3 to Server 1
Client: Request 3 completed by Server 1 in 0.30s
Result for request 3: Response from Server 1 for request 3, Latency: 0.30s (from Server 1)


## Circuit Breaker Pattern

### Concept Explanation
The Circuit Breaker pattern is a design pattern used in distributed systems to prevent a client from repeatedly trying to execute an operation that is likely to fail. This is particularly useful when interacting with remote services or resources. Just like an electrical circuit breaker prevents damage from overcurrent, a software circuit breaker stops requests from flowing to a failing service, protecting both the client (from long timeouts and resource exhaustion) and the service (from being overloaded by failing requests). The circuit breaker has three states:

1.  **Closed:** The circuit breaker is in its normal state. Requests are allowed to pass through to the service. If a request fails, the circuit breaker counts the failure. If the failure rate exceeds a certain threshold, the circuit breaker trips and moves to the Open state.
2.  **Open:** The circuit breaker is open. Requests to the service are immediately rejected or fail fast without attempting to call the service. After a configured timeout period, the circuit breaker moves to the Half-Open state.
3.  **Half-Open:** The circuit breaker allows a limited number of test requests to pass through to the service. If these test requests succeed, the circuit breaker assumes the service has recovered and moves back to the Closed state. If they fail, it reverts to the Open state for another timeout period.

### Real-World Analogy
Imagine you're trying to call a friend, but their phone keeps going straight to voicemail (the service is failing).

*   **Closed:** Initially, you just dial their number (requests pass through). If it goes to voicemail a few times in a row (failure rate exceeds threshold), you stop calling immediately.
*   **Open:** You decide not to call them for a while (requests are blocked). After a set time (timeout), you decide to try again cautiously.
*   **Half-Open:** You try calling *once* (a limited test request). If they answer, great! You assume their phone is working again and go back to calling normally (move to Closed). If it goes straight to voicemail again, you assume it's still broken and stop calling for another period (revert to Open).

This prevents you from wasting time and effort repeatedly calling a non-responsive number and gives your friend's phone a break if the issue is on their end.

### Use Cases & Trade-offs
**Use Cases:**
*   Protecting microservices from cascading failures when dependent services are unhealthy.
*   Interacting with external APIs or third-party services that may experience downtime or high latency.
*   Improving the resilience of applications by implementing graceful degradation when dependencies fail.

**Trade-offs:**
*   **Advantages:**
    *   Prevents cascading failures in distributed systems.
    *   Provides fail-fast behavior, improving client responsiveness.
    *   Gives failing services time to recover without being overloaded by continuous requests.
    *   Improves overall system stability and resilience.
*   **Disadvantages:**
    *   Adds complexity to the client-side logic.
    *   Requires careful configuration of failure thresholds and timeouts.
    *   The Half-Open state logic needs to be designed to avoid overwhelming a recovering service.
    *   Doesn't solve the underlying issue of the failing service itself.


In [42]:
import time
import random

class ServiceUnavailableError(Exception):
    """Custom exception for simulating service failures."""
    pass

class CircuitBreaker:
    def __init__(self, failure_threshold, recovery_timeout, half_open_test_requests):
        self.failure_threshold = failure_threshold # Number of failures before tripping
        self.recovery_timeout = recovery_timeout # Time to wait in Open state
        self.half_open_test_requests = half_open_test_requests # Requests allowed in Half-Open

        self.state = "CLOSED" # States: CLOSED, OPEN, HALF-OPEN
        self.failure_count = 0
        self.last_failure_time = None
        self.half_open_requests_attempted = 0
        self._lock = threading.Lock() # For thread-safe state changes

    def call(self, service_call):
        """Attempts to call the protected service."""
        with self._lock:
            current_state = self.state

        if current_state == "OPEN":
            # Check if recovery timeout has passed
            if self.last_failure_time and (time.time() - self.last_failure_time > self.recovery_timeout):
                print("Circuit Breaker: Timeout passed, moving to HALF-OPEN.")
                with self._lock:
                    self.state = "HALF-OPEN"
                    self.half_open_requests_attempted = 0
                return self.call(service_call) # Retry the call in Half-Open
            else:
                print("Circuit Breaker: OPEN. Request rejected.")
                return None # Fail fast

        elif current_state == "HALF-OPEN":
            with self._lock:
                if self.half_open_requests_attempted < self.half_open_test_requests:
                    self.half_open_requests_attempted += 1
                    print(f"Circuit Breaker: HALF-OPEN. Attempting test request {self.half_open_requests_attempted}/{self.half_open_test_requests}.")
                    try:
                        result = service_call()
                        # If successful in HALF-OPEN
                        print("Circuit Breaker: HALF-OPEN test successful. Moving to CLOSED.")
                        with self._lock:
                            self.state = "CLOSED"
                            self.failure_count = 0
                            self.half_open_requests_attempted = 0
                        return result
                    except ServiceUnavailableError:
                        # If failed in HALF-OPEN
                        print("Circuit Breaker: HALF-OPEN test failed. Moving back to OPEN.")
                        with self._lock:
                            self.state = "OPEN"
                            self.last_failure_time = time.time()
                            self.half_open_requests_attempted = 0
                        return None # Fail fast
                else:
                    print("Circuit Breaker: HALF-OPEN capacity reached. Request rejected.")
                    return None # Fail fast

        elif current_state == "CLOSED":
            print("Circuit Breaker: CLOSED. Allowing request.")
            try:
                result = service_call()
                # If successful in CLOSED, reset failure count
                with self._lock:
                    self.failure_count = 0
                return result
            except ServiceUnavailableError:
                # If failed in CLOSED
                with self._lock:
                    self.failure_count += 1
                    print(f"Circuit Breaker: Service call failed. Failure count: {self.failure_count}/{self.failure_threshold}.")
                    if self.failure_count >= self.failure_threshold:
                        print("Circuit Breaker: Failure threshold reached. Tripping to OPEN.")
                        self.state = "OPEN"
                        self.last_failure_time = time.time()
                        self.half_open_requests_attempted = 0
                return None # Indicate failure


# Simulate a flaky service that fails sometimes
def flaky_service(fail_rate=0.5):
    if random.random() < fail_rate:
        print("Service: Request failed!")
        raise ServiceUnavailableError("Service is unavailable")
    else:
        print("Service: Request successful!")
        return "Service Response"

# Example Usage
# Circuit breaker trips after 3 failures, waits 5 seconds in OPEN,
# allows 2 test requests in HALF-OPEN
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=5, half_open_test_requests=2)

print("--- Circuit Breaker Pattern Simulation ---")

# Simulate calls while service is healthy (initially, or with low fail rate)
print("\nSimulating calls when service is mostly healthy:")
for i in range(5):
    print(f"\nAttempt {i+1}:")
    result = breaker.call(lambda: flaky_service(fail_rate=0.2)) # Low fail rate
    if result:
        print("Received result:", result)
    else:
        print("Call failed or rejected.")
    time.sleep(0.1)

# Simulate calls when service starts failing frequently
print("\nSimulating calls when service starts failing frequently:")
for i in range(10):
    print(f"\nAttempt {i+1}:")
    result = breaker.call(lambda: flaky_service(fail_rate=0.8)) # High fail rate
    if result:
        print("Received result:", result)
    else:
        print("Call failed or rejected.")
    time.sleep(0.1)

# Simulate waiting for recovery timeout and then HALF-OPEN state
print(f"\nWaiting {breaker.recovery_timeout} seconds for recovery timeout...")
time.sleep(breaker.recovery_timeout + 1) # Wait a bit longer than the timeout

print("\nSimulating calls after recovery timeout (entering HALF-OPEN):")
for i in range(5):
    print(f"\nAttempt {i+1}:")
    # Service is still flaky, so half-open tests might fail
    result = breaker.call(lambda: flaky_service(fail_rate=0.6)) # Still some failures
    if result:
        print("Received result:", result)
    else:
        print("Call failed or rejected.")
    time.sleep(0.5) # Add some delay between attempts in HALF-OPEN/OPEN

# Simulate service recovery and successful transition back to CLOSED
print("\nSimulating calls when service has recovered:")
breaker.state = "HALF-OPEN" # Manually set back to HALF-OPEN for demonstration
print("Manually setting circuit breaker to HALF-OPEN for recovery test.")
for i in range(5):
    print(f"\nAttempt {i+1}:")
    result = breaker.call(lambda: flaky_service(fail_rate=0.1)) # Low fail rate (recovered)
    if result:
        print("Received result:", result)
    else:
        print("Call failed or rejected.")
    time.sleep(0.2)

print("\nSimulation finished.")

--- Circuit Breaker Pattern Simulation ---

Simulating calls when service is mostly healthy:

Attempt 1:
Circuit Breaker: CLOSED. Allowing request.
Service: Request successful!
Received result: Service Response

Attempt 2:
Circuit Breaker: CLOSED. Allowing request.
Service: Request failed!
Circuit Breaker: Service call failed. Failure count: 1/3.
Call failed or rejected.

Attempt 3:
Circuit Breaker: CLOSED. Allowing request.
Service: Request successful!
Received result: Service Response

Attempt 4:
Circuit Breaker: CLOSED. Allowing request.
Service: Request successful!
Received result: Service Response

Attempt 5:
Circuit Breaker: CLOSED. Allowing request.
Service: Request failed!
Circuit Breaker: Service call failed. Failure count: 1/3.
Call failed or rejected.

Simulating calls when service starts failing frequently:

Attempt 1:
Circuit Breaker: CLOSED. Allowing request.
Service: Request failed!
Circuit Breaker: Service call failed. Failure count: 2/3.
Call failed or rejected.

Attem

KeyboardInterrupt: 